# LAP01 · Multitracking multicámara (AeroVision)

Un solo notebook con las **Partes I y II** de la metodología aplicadas a **tres cámaras físicas distintas** (`cam01`, `cam02`, `cam03`): cada una ve el recinto desde un ángulo diferente.

```
CCTV → Tracking local (LAP01) → Mapeo 2D (homografía) → Re-ID multicámara → trajectory_points
```

| Carpeta | Contenido |
|---|---|
| `dataset/` | Videos de las tres cámaras (`cam0X_recortado_1280x720_15fps.mp4`). |
| `Modelo/` | Lo que usa el modelo: `yolo26m.pt`, `resnet18-f37072fd.pth`, `yolo26s-reid.onnx`, `config_lap01.json`, `camaras.json` y el manifiesto `exportacion.json`. |
| `Ouput/` | Resultados: un video procesado por cámara y `trajectory_points.csv`. |

**Parte I (por cámara):** YOLO26m detecta personas → limpieza de duplicados → predicción de movimiento, IoU y escala → apariencia sin rostro (HSV/Lab por cabeza, torso y piernas + ResNet18) → memoria visual multivista → Hungarian con opción de no asignar → oclusiones y recuperación → `local_id`.

**Parte II (entre cámaras):** registro técnico de cámaras → sincronización (`timestamp_offset`) → homografía al plano (si está calibrada) → tracklets → grafo de conectividad y gating físico → similitud híbrida (apariencia, tiempo, posición, dirección y velocidad) → `global_id` anónimo.

**Fuera de alcance por ahora:** zonas, eventos espaciales (`zones`, `spatial_events`) y Parte III. En `trajectory_points`, `zone_id` queda `NULL`.

No se entrena ni se descarga nada: todos los pesos se leen de `Modelo/`.

## 1. Entorno y rutas

Usa el kernel de **jupyterev**. Ultralytics queda sin descargas, autoinstalación ni telemetría, y guarda su configuración en `Modelo/.runtime/`. Si ya habías importado Ultralytics en este kernel, reinícialo antes de ejecutar esta celda.

In [ ]:
from pathlib import Path
import os
import sys

# Carpeta del proyecto: dataset/ (videos), Modelo/ (pesos y configuración) y Ouput/ (resultados).
BASE = next((carpeta.resolve() for carpeta in (Path.cwd(), Path.cwd() / "02_multicamara")
             if (carpeta / "Modelo" / "config_lap01.json").is_file() and (carpeta / "dataset").is_dir()), None)
if BASE is None:
    raise FileNotFoundError("Abre JupyterLab desde 02_multicamara o desde Entrenamiento.")
if Path(sys.prefix).name != "jupyterev":
    raise RuntimeError("Selecciona el kernel del entorno jupyterev antes de continuar.")

DATASET = BASE / "dataset"
MODELO = BASE / "Modelo"
OUTPUT = BASE / "Ouput"
OUTPUT.mkdir(exist_ok=True)
NOTEBOOK = BASE / "LAP01_Multitracking.ipynb"

# Deben fijarse antes de importar Ultralytics.
if "ultralytics" in sys.modules:
    print("Ultralytics ya estaba importado: reinicia el kernel para aplicar la configuración local.")
os.environ["YOLO_CONFIG_DIR"] = str(MODELO / ".runtime" / "ultralytics")
os.environ["YOLO_OFFLINE"] = "1"
os.environ["YOLO_AUTOINSTALL"] = "0"

import hashlib
import heapq
import importlib.metadata
import json
import math
import time
import uuid
from collections import deque
from contextlib import ExitStack
from copy import deepcopy
from dataclasses import dataclass, field
from datetime import datetime
from functools import lru_cache

import cv2
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display
from scipy.optimize import linear_sum_assignment
from torch import nn
from threadpoolctl import threadpool_limits

print("Python:", sys.executable)
print("Carpeta:", BASE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no disponible (CPU)")

## 2. Configurar la prueba

Cada archivo `dataset/cam0X_*.mp4` es una cámara distinta. Sus identificadores deben coincidir con `Modelo/camaras.json`.

- `MAX_FRAMES = None` procesa los videos completos (usa `150` para una prueba de 10 s). No se saltan frames.
- `area` es la zona con imagen real de cada cámara: las bandas negras del recorte no cuentan como borde de la escena.
- `INICIOS` recorta el principio de cada video sin cambiar su reloj.
- **Tiempo común = tiempo del video + `timestamp_offset`** (en `camaras.json`). Un offset positivo retrasa la cámara.
- `INICIO_GRABACION` convierte ese tiempo relativo en el `TIMESTAMPTZ` de `trajectory_points`. Escribe la hora real de la grabación. Con `None` se usa la hora en que empieza la sesión.

In [ ]:
VIDEOS = {ruta.name.split("_")[0]: ruta for ruta in sorted(DATASET.glob("cam*.mp4"))}

MAX_FRAMES = None                       # Videos completos; p. ej. 150 = 10 s por cámara para una prueba corta.
INICIOS = {cid: 0 for cid in VIDEOS}    # Frame inicial de cada cámara (base cero).
TIEMPO_REAL = True                      # Limita el ritmo al del video si el procesamiento va más rápido.
VISTA_EN_VIVO = True                    # Mosaico de las cámaras mientras se procesa.
VISTA_FPS = 8.0
VISTA_ANCHO = 960                       # Ancho de cada cámara en el mosaico.
DEVICE = "auto"                         # cuda:0 si hay GPU; si no, CPU.
BATCH_YOLO = True                       # Un lote YOLO con un frame de cada cámara.
CPU_THREADS = 2
INICIO_GRABACION = None                 # p. ej. "2026-09-12T14:30:00-05:00"

EXPORTAR_VIDEOS = True                  # Ouput/<cam>_procesado.mp4 con los global_id finales.
EXPORTAR_TRAYECTORIAS = True            # Ouput/trajectory_points.csv para PostGIS.

CONFIG = json.loads((MODELO / "config_lap01.json").read_text())
CONFIG_CAMARAS = json.loads((MODELO / "camaras.json").read_text())


def area_util(cap, frames, umbral=8.0):
    """Zona con imagen real (sin bandas negras del recorte): (x0, y0, x1, y1)."""
    muestras = []
    for k in (0, frames // 2, (2 * frames) // 3):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(k))
        ok, frame = cap.read()
        if ok:
            muestras.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    if not muestras:
        raise RuntimeError("No se pudieron leer frames para medir el área útil.")
    gris = np.max(np.stack(muestras), axis=0).astype(np.float32)
    filas, columnas = np.where(gris.mean(axis=1) > umbral)[0], np.where(gris.mean(axis=0) > umbral)[0]
    if not len(filas) or not len(columnas):
        return (0, 0, gris.shape[1], gris.shape[0])
    return (int(columnas[0]), int(filas[0]), int(columnas[-1]) + 1, int(filas[-1]) + 1)


def inspeccionar_videos(videos):
    """Lee metadatos y el área útil sin cargar los videos en memoria."""
    if not videos:
        raise FileNotFoundError(f"No hay videos cam*.mp4 en {DATASET}")
    filas = []
    for camera_id, ruta in videos.items():
        ruta = Path(ruta).resolve()
        cap = cv2.VideoCapture(str(ruta))
        try:
            if not cap.isOpened():
                raise RuntimeError(f"No se pudo abrir {ruta}")
            fps = float(cap.get(cv2.CAP_PROP_FPS))
            frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if not math.isfinite(fps) or fps <= 0 or frames <= 0:
                raise ValueError(f"Metadatos inválidos en {ruta}")
            filas.append({"camera_id": camera_id, "video": str(ruta), "fps": fps, "frames": frames,
                          "duration_s": frames / fps,
                          "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                          "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                          "area": area_util(cap, frames)})
        finally:
            cap.release()
    return filas


if set(VIDEOS) != set(CONFIG_CAMARAS["cameras"]):
    raise ValueError(f"Cámaras del dataset {sorted(VIDEOS)} ≠ Modelo/camaras.json {sorted(CONFIG_CAMARAS['cameras'])}.")

display(pd.DataFrame(inspeccionar_videos(VIDEOS)))
print("Modo multicámara:", CONFIG_CAMARAS["mode"])
display(pd.DataFrame({cid: {"timestamp_offset": cam.get("timestamp_offset", 0.0),
                            "resolution": cam.get("resolution"),
                            "homografia": cam.get("H") is not None}
                      for cid, cam in CONFIG_CAMARAS["cameras"].items()}).T)

# Parte I — Seguimiento local (LAP01)

Cada cámara ejecuta su propio seguidor y respeta el orden temporal de sus frames. El `local_id` solo es válido dentro de la cámara que lo creó.

### 3.1 — Detección de personas

YOLO26m restringido a la clase `person`. Cada detección es $d_i=(x_1,y_1,x_2,y_2,c_i)$. Una confianza baja (`conf`) solo sirve para continuar un track existente; nunca crea una identidad.

In [ ]:
class DetectorPersonas:
    """YOLO26m, clase person. Con batch=True agrupa un frame por cámara (nunca frames de distinto tiempo)."""

    def __init__(self, pesos, config, device, batch=True):
        from ultralytics import YOLO

        self.model = YOLO(str(pesos))
        self.config, self.device, self.batch = config, device, batch

    def __call__(self, frames):
        # FP16 es ~35 % más rápido y da las mismas cajas, pero sus confianzas cambian en milésimas y eso alteró
        # decisiones del tracker en el umbral (precisión multicámara 0.99 -> 0.95). Por defecto: FP32.
        precision = 16 if self.config.get("fp16", False) and str(self.device).startswith("cuda") else None
        kwargs = dict(classes=[0], conf=self.config["conf"], iou=self.config["iou"], imgsz=self.config["imgsz"],
                      device=self.device, verbose=False, rect=True, quantize=precision)
        images = list(frames.values())
        results = (self.model.predict(source=images, **kwargs) if self.batch
                   else [self.model.predict(source=image, **kwargs)[0] for image in images])
        detections = {}
        for cid, result in zip(frames, results):
            if result.boxes is None or not len(result.boxes):
                detections[cid] = (np.empty((0, 4), np.float32), np.empty(0, np.float32))
            else:
                detections[cid] = (result.boxes.xyxy.cpu().numpy().astype(np.float32),
                                   result.boxes.conf.cpu().numpy().astype(np.float32))
        return detections

### 3.2 — Apariencia sin reconocimiento facial

El recorte se divide en **cabeza, torso y piernas**. De cada región visible se obtienen histogramas HSV y Lab. Las zonas tapadas por otra persona se enmascaran para no mezclar identidades. ResNet18 (pesos locales de ImageNet) se usa solo como extractor profundo:

$$A=[A_{color},A_{deep}]$$

La mezcla calibrada es `clip(gain · ((1−w)·color + w·deep) + offset, 0, 1)`, con `w`, `gain` y `offset` tomados de `config_lap01.json`.

**Memoria visual multivista:** cada identidad guarda una galería de vistas fiables y se compara con $S_{app}(q,G_i)=\max_{g\in G_i}\operatorname{sim}(q,g)$, en forma vectorizada.

In [ ]:
_REGIONS = ((slice(0, 120), 0.20), (slice(120, 240), 0.52), (slice(240, 360), 0.28))  # cabeza, torso, piernas
_PESOS_COLOR = np.concatenate([np.full(120, weight, np.float32) for _, weight in _REGIONS])


def _l1_sqrt(hist):
    hist = hist.astype(np.float32).reshape(-1)
    return np.sqrt(hist / (hist.sum() + 1e-8))


def _part_signature(part, mask=None):
    """Histogramas HSV y Lab de una región corporal visible (120 valores)."""
    if part.size == 0 or (mask is not None and np.count_nonzero(mask) < 32):
        return np.zeros(120, dtype=np.float32)
    part = cv2.resize(part, (48, 64), interpolation=cv2.INTER_AREA)
    if mask is not None:
        mask = cv2.resize(mask.astype(np.uint8), (48, 64), interpolation=cv2.INTER_NEAREST)
    hsv = cv2.cvtColor(part, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(part, cv2.COLOR_BGR2LAB)
    hs = _l1_sqrt(cv2.calcHist([hsv], [0, 1], mask, [12, 4], [0, 180, 0, 256]))
    value = _l1_sqrt(cv2.calcHist([hsv], [2], mask, [8], [0, 256]))
    chroma = _l1_sqrt(cv2.calcHist([lab], [1, 2], mask, [8, 8], [0, 256, 0, 256]))
    return np.concatenate((hs, value, chroma)).astype(np.float32)


def clothing_signature(image, box, all_boxes=None):
    """Firma de color de cabeza, torso y piernas excluyendo lo cubierto por otras cajas (360 valores)."""
    x1, y1, x2, y2 = map(int, box)
    height, width = image.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(width, x2), min(height, y2)
    crop = image[y1:y2, x1:x2]
    if crop.shape[0] < 16 or crop.shape[1] < 8:
        return np.zeros(360, dtype=np.float32)

    safe_mask = np.full(crop.shape[:2], 255, dtype=np.uint8)
    if all_boxes is not None:
        for other in np.asarray(all_boxes, dtype=np.float32):
            if np.allclose(other, np.asarray(box, dtype=np.float32), atol=1.0):
                continue
            ox1, oy1, ox2, oy2 = map(int, other)
            ix1, iy1 = max(x1, ox1), max(y1, oy1)
            ix2, iy2 = min(x2, ox2), min(y2, oy2)
            if ix2 > ix1 and iy2 > iy1:
                safe_mask[iy1 - y1:iy2 - y1, ix1 - x1:ix2 - x1] = 0

    h, w = crop.shape[:2]
    head_top, head_bottom = int(0.02 * h), int(0.28 * h)
    head_left, head_right = int(0.22 * w), int(0.78 * w)
    head = _part_signature(crop[head_top:head_bottom, head_left:head_right],
                           safe_mask[head_top:head_bottom, head_left:head_right])

    top, bottom = int(0.20 * h), int(0.94 * h)
    left, right = int(0.16 * w), int(0.84 * w)
    body, body_mask = crop[top:bottom, left:right], safe_mask[top:bottom, left:right]
    split = max(1, int(body.shape[0] * 0.55))
    signature = np.concatenate((head, _part_signature(body[:split], body_mask[:split]),
                                _part_signature(body[split:], body_mask[split:])))
    return signature / (np.linalg.norm(signature) + 1e-8)


def _unidad_por_region(colours):
    """Normaliza cabeza, torso y piernas por separado. (unidad · pesos) @ unidad = similitud de color."""
    colours = np.atleast_2d(np.asarray(colours, dtype=np.float32))
    unit = np.empty_like(colours)
    for region, _ in _REGIONS:
        block = colours[:, region]
        unit[:, region] = block / (np.linalg.norm(block, axis=1, keepdims=True) + 1e-8)
    return unit


def colour_similarity(first, second):
    return float((_unidad_por_region(first)[0] * _PESOS_COLOR) @ _unidad_por_region(second)[0])


class ExtractorResNet18:
    """ResNet18 sin capa de clasificación; pesos locales, sin descargas. Devuelve 2048 valores normalizados."""
    ENTRADA = (64, 128)  # ancho, alto

    def __init__(self, pesos, device):
        from torchvision.models import resnet18

        network = resnet18(weights=None)
        network.load_state_dict(torch.load(pesos, map_location="cpu", weights_only=True))
        self.device = device
        self.model = torch.nn.Sequential(*list(network.children())[:-2]).to(device).eval()
        self.mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    @torch.inference_mode()
    def __call__(self, image, boxes):
        if not len(boxes):
            return []
        height, width = image.shape[:2]
        crops = []
        for box in boxes:
            x1, y1, x2, y2 = (int(value) for value in box)
            crop = image[max(0, y1):min(height, y2), max(0, x1):min(width, x2)]
            if crop.size == 0:
                crop = np.zeros((8, 4, 3), dtype=np.uint8)
            crops.append(cv2.resize(crop, self.ENTRADA, interpolation=cv2.INTER_LINEAR))
        batch = np.ascontiguousarray(np.stack(crops)[..., ::-1])
        tensor = torch.from_numpy(batch).to(self.device).permute(0, 3, 1, 2).float() / 255.0
        maps = self.model((tensor - self.mean) / self.std)
        pooled = torch.nn.functional.adaptive_avg_pool2d(maps, (4, 1)).flatten(1)
        return list(torch.nn.functional.normalize(pooled, dim=1).cpu().numpy().astype(np.float32))


@dataclass(frozen=True)
class MezclaApariencia:
    deep_weight: float = 0.20
    gain: float = 1.123
    offset: float = -0.075

    def __call__(self, colour, deep):
        mixed = (1.0 - self.deep_weight) * colour + self.deep_weight * deep
        return np.clip(self.gain * mixed + self.offset, 0.0, 1.0)


@dataclass
class Appearance:
    colour: np.ndarray
    deep: np.ndarray | None = None
    _unit: np.ndarray | None = field(default=None, repr=False, compare=False)

    def unit(self):
        if self._unit is None:
            self._unit = _unidad_por_region(self.colour)[0]
        return self._unit

    def similarity(self, other, mezcla):
        colour = float((self.unit() * _PESOS_COLOR) @ other.unit())
        if self.deep is None or other.deep is None:
            return colour
        return float(mezcla(colour, float(self.deep @ other.deep)))


def gallery_similarities(signatures, tracks, mezcla):
    """Matriz tracks × detecciones con la mejor vista de la galería de cada track."""
    if not signatures or not tracks:
        return np.zeros((len(tracks), len(signatures)), dtype=np.float32)
    gallery, starts = [], []
    for track in tracks:
        starts.append(len(gallery))
        gallery.extend(track.gallery)
    similarity = np.stack([s.unit() for s in signatures]) @ (np.stack([v.unit() for v in gallery]) * _PESOS_COLOR).T
    if all(v.deep is not None for v in gallery) and all(s.deep is not None for s in signatures):
        deep = np.stack([s.deep for s in signatures]) @ np.stack([v.deep for v in gallery]).T
        similarity = mezcla(similarity, deep)
    return np.maximum.reduceat(similarity, starts, axis=1).T.astype(np.float32)


def describe_indices(image, boxes, indices, extractor=None):
    """Apariencia solo de algunas cajas; todas las demás siguen contando como oclusores en la máscara de color."""
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    indices = list(indices)
    deep = extractor(image, boxes[indices]) if extractor is not None and indices else [None] * len(indices)
    return {i: Appearance(clothing_signature(image, boxes[i], boxes), vector) for i, vector in zip(indices, deep)}

### 3.3 — Asociación, oclusiones y recuperación

Por frame, en este orden:

1. **Limpieza:** se descartan cajas contenidas en otra en un ≥ 92 % y cajas demasiado pequeñas.
2. **Asociación principal** (detecciones con confianza ≥ `association_confidence`, tracks vistos en el frame anterior): $S_{ij}=w_mS_{mov}+w_oS_{IoU}+w_sS_{escala}+w_aS_{app}$. El costo se resuelve con Hungarian y columnas ficticias, de modo que un track puede quedar sin asignar sin quitarle la detección a un par válido.
3. **Segunda etapa (tipo ByteTrack):** una detección débil solo prolonga un track activo libre con movimiento, IoU y escala muy coherentes.
4. **Recuperación:** las detecciones fiables no asignadas se comparan con tracks perdidos (≤ `max_age`) usando la galería de apariencia, la posición esperada y la escala.
5. **Actualización:** la velocidad se amortigua y la galería solo recibe vistas fiables (confianza ≥ 0.55, solape < 0.25 y evidencia de color).
6. **Tracks nuevos:** requieren `min_confirmations` detecciones dentro de `confirmation_window`. No se publican IDs efímeros.

**La apariencia no se calcula para cada persona en cada frame.** El color por partes y la ResNet18 solo se calculan cuando cambian una decisión:

- Una detección cerca de un track con poco IoU.
- Dos tracks o dos detecciones que compiten.
- Un track perdido que se intenta recuperar.
- Una persona nueva o tentativa.
- El refresco de la galería, cada `appearance_interval` frames.

Si movimiento, IoU y escala ya bastan para decidir, se omite.

Medido en los videos completos: con `appearance_interval = 1` el resultado es idéntico al cálculo en todos los frames (IDF1 = 1.000 en las tres cámaras). Con `5` se calcula la apariencia solo para el **23 %** de las detecciones y la asociación multicámara queda igual (precisión 0.99, recall 0.86).

In [ ]:
def _has_colour_evidence(appearance):
    return bool(np.linalg.norm(appearance.colour) > 0.20)


def _center_distance(first, second):
    dx = (first[0] + first[2] - second[0] - second[2]) / 2.0
    dy = (first[1] + first[3] - second[1] - second[3]) / 2.0
    return float((dx * dx + dy * dy) ** 0.5)


def _scale_change(first, second):
    first_w, first_h = max(1.0, first[2] - first[0]), max(1.0, first[3] - first[1])
    second_w, second_h = max(1.0, second[2] - second[0]), max(1.0, second[3] - second[1])
    return float((abs(np.log(second_w / first_w)) + abs(np.log(second_h / first_h))) / 2.0)


def _iou(first, second):
    x1, y1 = max(first[0], second[0]), max(first[1], second[1])
    x2, y2 = min(first[2], second[2]), min(first[3], second[3])
    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_first = max(0.0, first[2] - first[0]) * max(0.0, first[3] - first[1])
    area_second = max(0.0, second[2] - second[0]) * max(0.0, second[3] - second[1])
    return float(intersection / (area_first + area_second - intersection + 1e-8))


def _asignar_costos(costs, threshold):
    """Hungarian con columnas ficticias: permite dejar filas sin asignar sin robar pares válidos."""
    augmented = np.full((len(costs), costs.shape[1] + len(costs)), threshold + 1e-6)
    augmented[:, :costs.shape[1]] = np.where(costs <= threshold, costs, 1e6)
    rows, cols = linear_sum_assignment(augmented)
    valid = cols < costs.shape[1]
    return rows[valid], cols[valid]


def _nested_duplicate_indices(boxes, scores):
    """Cajas contenidas en otra en un 92 % o más: se conserva la de mayor confianza."""
    dropped = set()
    for first in range(len(boxes)):
        if first in dropped:
            continue
        for second in range(first + 1, len(boxes)):
            if second in dropped:
                continue
            left, top = max(boxes[first, 0], boxes[second, 0]), max(boxes[first, 1], boxes[second, 1])
            right, bottom = min(boxes[first, 2], boxes[second, 2]), min(boxes[first, 3], boxes[second, 3])
            intersection = max(0.0, right - left) * max(0.0, bottom - top)
            area_first = max(1.0, (boxes[first, 2] - boxes[first, 0]) * (boxes[first, 3] - boxes[first, 1]))
            area_second = max(1.0, (boxes[second, 2] - boxes[second, 0]) * (boxes[second, 3] - boxes[second, 1]))
            if intersection / min(area_first, area_second) >= 0.92:
                dropped.add(first if scores[first] < scores[second] else second)
    return dropped


@dataclass
class _StableTrack:
    stable_id: int
    box: np.ndarray
    appearance: Appearance
    frame_seen: int
    velocity: np.ndarray = field(default_factory=lambda: np.zeros(4, dtype=np.float32))
    gallery: deque = field(default_factory=lambda: deque(maxlen=24))
    gallery_frame: int = -1  # último frame en que se revisó su apariencia


@dataclass
class _TentativeTrack:
    box: np.ndarray
    appearance: Appearance
    first_seen: int
    frame_seen: int
    hits: int = 1
    velocity: np.ndarray = field(default_factory=lambda: np.zeros(4, dtype=np.float32))


class ColorOcclusionTracker:
    """Seguidor local LAP01: movimiento, IoU, escala, apariencia y memoria visual por cámara."""

    def __init__(self, extractor, mezcla, max_age=120, colour_gate=0.72, distance_gate=0.95,
                 gallery_size=24, deep_appearance=True, min_confirmations=3, confirmation_window=5,
                 new_track_confidence=0.45, association_confidence=0.35, active_motion_gate=0.55,
                 active_iou_gate=0.12, min_person_height=18, reid_gate=0.80, appearance_interval=5):
        if (min_confirmations < 1 or confirmation_window < min_confirmations or max_age < 1 or gallery_size < 1
                or appearance_interval < 1):
            raise ValueError("Ventanas de tracking o tamaño de galería inválidos.")
        self.extractor = extractor if deep_appearance else None
        self.mezcla = mezcla
        self.max_age = max_age
        self.colour_gate = colour_gate
        self.distance_gate = distance_gate
        self.gallery_size = gallery_size
        self.min_confirmations = min_confirmations
        self.confirmation_window = confirmation_window
        self.new_track_confidence = new_track_confidence
        self.association_confidence = association_confidence
        self.active_motion_gate = active_motion_gate
        self.active_iou_gate = active_iou_gate
        self.min_person_height = min_person_height
        self.reid_gate = reid_gate
        self.appearance_interval = appearance_interval
        self.last_frame = None
        self.tracks = {}
        self.tentative = []
        self.riesgo = set()  # IDs recuperados tras una pérdida o asociados en un cruce durante el último frame
        self.next_stable_id = 1
        self.diagnostics = {"low_quality_rejected": 0, "duplicates_suppressed": 0, "tracks_confirmed": 0,
                            "reid_matches": 0, "short_occlusion_recoveries": 0, "low_confidence_updates": 0,
                            "detections": 0, "appearance_computed": 0}
        self._imagen, self._cajas, self._firmas = None, np.empty((0, 4), np.float32), {}

    @staticmethod
    def _predicted_box(track, frame):
        gap = max(0, frame - track.frame_seen)
        return track.box + track.velocity * ((1.0 - 0.84 ** min(gap, 8)) / (1.0 - 0.84))

    def _appearance_similarity(self, signature, track):
        return max(signature.similarity(view, self.mezcla) for view in track.gallery)

    def _expire(self, frame):
        for track_id in [tid for tid, track in self.tracks.items() if frame - track.frame_seen > self.max_age]:
            del self.tracks[track_id]
        self.tentative = [track for track in self.tentative
                          if frame - track.frame_seen < self.confirmation_window
                          and frame - track.first_seen < self.confirmation_window]

    def _firmas_de(self, indices):
        """Apariencia de las detecciones indicadas del frame actual; cada una se calcula como mucho una vez."""
        faltan = [i for i in indices if i not in self._firmas]
        if faltan:
            self._firmas.update(describe_indices(self._imagen, self._cajas, faltan, self.extractor))
            self.diagnostics["appearance_computed"] += len(faltan)
        return [self._firmas[i] for i in indices]

    def _update_track(self, track, index, frame, score, max_overlap):
        box = self._cajas[index]
        step = max(1, frame - track.frame_seen)
        track.velocity = 0.72 * track.velocity + 0.28 * (box - track.box) / step
        # La galería solo acepta vistas fiables. Se revisa si la apariencia ya se calculó en este frame o si
        # pasaron `appearance_interval` frames; el resto de frames no se procesa la apariencia de la persona.
        revisar = index in self._firmas or frame - track.gallery_frame >= self.appearance_interval
        if revisar and score >= 0.55 and max_overlap < 0.25:
            signature = self._firmas_de([index])[0]
            visual = self._appearance_similarity(signature, track)
            if _has_colour_evidence(signature) and visual >= 0.65:
                if visual < 0.97:
                    track.gallery.append(signature)  # vista nueva y fiable: amplía la memoria multivista
                track.appearance = signature
            track.gallery_frame = frame
        track.box = box
        track.frame_seen = frame

    def _geometria(self, predicted, box):
        size = max(np.hypot(predicted[2] - predicted[0], predicted[3] - predicted[1]),
                   np.hypot(box[2] - box[0], box[3] - box[1]), 30.0)
        return _center_distance(predicted, box) / size, _iou(predicted, box), _scale_change(predicted, box)

    def _match_confirmed(self, indices, frame, candidates, strict_low=False):
        if not candidates or not indices:
            return {}
        boxes = self._cajas[indices]
        n_tracks, n_dets = len(candidates), len(indices)
        motion, overlap, scale = (np.zeros((n_tracks, n_dets), np.float32) for _ in range(3))
        gaps = np.array([max(1, frame - track.frame_seen) for track in candidates])
        gates = np.minimum(2.3, self.distance_gate * (1.0 + 0.18 * (gaps - 1)))
        for row, track in enumerate(candidates):
            predicted = self._predicted_box(track, frame)
            for column, box in enumerate(boxes):
                motion[row, column], overlap[row, column], scale[row, column] = self._geometria(predicted, box)
        costs = np.full((n_tracks, n_dets), 1e6, dtype=np.float32)

        if strict_low:
            # Una caja débil no puede cambiar una identidad: solo continuidad casi exacta, sin apariencia.
            valid = (gaps[:, None] == 1) & (motion <= 0.35) & (overlap >= 0.20) & (scale <= 0.65)
            costs[valid] = (0.62 * np.minimum(motion / 0.35, 1.0) + 0.28 * (1.0 - overlap)
                            + 0.10 * np.minimum(scale / 0.65, 1.0))[valid]
        else:
            activo = gaps[:, None] == 1
            base = (motion <= np.minimum(gates, self.active_motion_gate)[:, None]) & (scale <= 0.85)
            seguro = activo & base & (overlap >= self.active_iou_gate)                     # válido sin apariencia
            depende = (activo & base & (overlap < self.active_iou_gate) & (motion <= 0.35)) | (
                ~activo & (motion <= gates[:, None]))                                       # decide la apariencia
            posible = seguro | depende
            geometria = (0.42 * np.minimum(motion / gates[:, None], 1.0) + 0.22 * (1.0 - overlap)
                         + 0.14 * np.minimum(scale / 0.85, 1.0))
            # La apariencia solo se calcula si cambia la decisión: validez dudosa, competencia entre
            # tracks/detecciones o un par aislado que con la peor apariencia superaría el costo máximo.
            competencia = posible & ((posible.sum(1) > 1)[:, None] | (posible.sum(0) > 1)[None, :])
            necesita = (depende | competencia | (seguro & (geometria + 0.22 > 0.72))).any(axis=0)
            columnas = [int(c) for c in np.flatnonzero(necesita)]
            firmas = self._firmas_de([indices[c] for c in columnas])
            parecido = gallery_similarities(firmas, candidates, self.mezcla)
            for k, column in enumerate(columnas):
                con_color = _has_colour_evidence(firmas[k])
                visual = parecido[:, k] if con_color else np.full(n_tracks, 0.50, np.float32)
                for row in np.flatnonzero(posible[:, column]):
                    if gaps[row] == 1:
                        ok = seguro[row, column] or (con_color and visual[row] >= 0.78)
                    else:
                        continuidad = (gaps[row] <= 8 and motion[row, column] <= 0.35
                                       and overlap[row, column] >= 0.20 and scale[row, column] <= 0.70)
                        requerido = min(0.88, self.reid_gate + 0.01 * min(gaps[row] - 2, 8))
                        ok = con_color and (visual[row] >= requerido or (continuidad and visual[row] >= 0.56))
                    if ok:
                        costs[row, column] = geometria[row, column] + 0.22 * (1.0 - visual[row])
            aislados = seguro & ~necesita[None, :]
            costs[aislados] = geometria[aislados] + 0.22  # par único: la apariencia no cambia la asignación

        matches = {}
        for row, column in zip(*_asignar_costos(costs, 0.72)):
            track = candidates[row]
            matches[int(column)] = track.stable_id
            if frame - track.frame_seen > 1:
                self.riesgo.add(track.stable_id)
                self.diagnostics["reid_matches"] += 1
                if motion[row, column] <= 0.35 and overlap[row, column] >= 0.20:
                    self.diagnostics["short_occlusion_recoveries"] += 1
        return matches

    def _match_tentative(self, indices, frame):
        if not self.tentative or not indices:
            return {}
        boxes = self._cajas[indices]
        motion = np.array([[self._geometria(self._predicted_box(track, frame), box)[0] for box in boxes]
                           for track in self.tentative], dtype=np.float32)
        cercanas = [int(c) for c in np.flatnonzero((motion <= 1.05).any(axis=0))]
        costs = np.full((len(self.tentative), len(indices)), 1e6, dtype=np.float32)
        for column, signature in zip(cercanas, self._firmas_de([indices[c] for c in cercanas])):
            for row, track in enumerate(self.tentative):
                appearance = signature.similarity(track.appearance, self.mezcla)
                if motion[row, column] <= 1.05 and appearance >= 0.58:
                    costs[row, column] = 0.60 * motion[row, column] / 1.05 + 0.40 * (1.0 - appearance)
        rows, columns = _asignar_costos(costs, 0.66)
        return {int(column): int(row) for row, column in zip(rows, columns)}

    def update(self, image, boxes, frame, scores=None):
        """Devuelve un local_id por caja (-1 si la caja no publica identidad en este frame)."""
        boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
        scores = np.ones(len(boxes), np.float32) if scores is None else np.asarray(scores, dtype=np.float32)
        if len(boxes) != len(scores):
            raise ValueError("boxes y scores deben tener la misma longitud.")
        if self.last_frame is not None and frame <= self.last_frame:
            raise ValueError("Los frames de una cámara deben ser estrictamente crecientes.")
        self.last_frame = frame
        self.riesgo = set()
        self._expire(frame)
        ids = [-1] * len(boxes)
        if not len(boxes):
            return ids

        duplicates = _nested_duplicate_indices(boxes, scores)
        valid_indices = []
        for index, box in enumerate(boxes):
            if index in duplicates:
                self.diagnostics["duplicates_suppressed"] += 1
            elif box[2] - box[0] < 6 or box[3] - box[1] < self.min_person_height:
                self.diagnostics["low_quality_rejected"] += 1
            else:
                valid_indices.append(index)
        if not valid_indices:
            return ids

        current_scores = scores[valid_indices]
        self._imagen, self._cajas, self._firmas = image, boxes[valid_indices], {}
        self.diagnostics["detections"] += len(valid_indices)

        # 1) Asociación principal: detecciones fiables contra tracks activos (vistos en el frame anterior).
        active_tracks = [track for track in self.tracks.values() if frame - track.frame_seen == 1]
        high = [i for i, score in enumerate(current_scores) if score >= self.association_confidence]
        matches = {}
        if high and active_tracks:
            found = self._match_confirmed(high, frame, candidates=active_tracks)
            matches.update({high[i]: stable_id for i, stable_id in found.items()})

        # 2) Detecciones débiles: solo prolongan un track activo que siga libre.
        low = [i for i, score in enumerate(current_scores) if score < self.association_confidence]
        available = [track for track in active_tracks if track.stable_id not in set(matches.values())]
        if low and available:
            found = self._match_confirmed(low, frame, candidates=available, strict_low=True)
            matches.update({low[i]: stable_id for i, stable_id in found.items()})
            self.diagnostics["low_confidence_updates"] += len(found)

        # 3) Recuperación: tracks perdidos contra detecciones fiables no reclamadas.
        recovery = [i for i in high if i not in matches]
        lost = [track for track in self.tracks.values()
                if 1 < frame - track.frame_seen <= self.max_age and track.stable_id not in set(matches.values())]
        if recovery and lost:
            found = self._match_confirmed(recovery, frame, candidates=lost)
            matches.update({recovery[i]: stable_id for i, stable_id in found.items()})

        # 4) Actualización del estado de los tracks asociados.
        for local_index, stable_id in matches.items():
            max_overlap = max((_iou(self._cajas[local_index], other)
                               for other_index, other in enumerate(self._cajas) if other_index != local_index),
                              default=0.0)
            if max_overlap >= 0.25:
                self.riesgo.add(stable_id)
            self._update_track(self.tracks[stable_id], local_index, frame, float(current_scores[local_index]), max_overlap)
            ids[valid_indices[local_index]] = stable_id

        # 5) Tracks tentativos: solo detecciones fiables pueden confirmar una identidad nueva.
        remaining = [i for i in range(len(self._cajas))
                     if i not in matches and current_scores[i] >= self.new_track_confidence]
        matched_tentative = set()
        for local_remaining, tentative_index in self._match_tentative(remaining, frame).items():
            local_index = remaining[local_remaining]
            track = self.tentative[tentative_index]
            step = max(1, frame - track.frame_seen)
            track.velocity = 0.70 * track.velocity + 0.30 * (self._cajas[local_index] - track.box) / step
            track.box = self._cajas[local_index]
            track.appearance = self._firmas_de([local_index])[0]
            track.frame_seen = frame
            track.hits += 1
            matched_tentative.add(local_index)
            if track.hits >= self.min_confirmations and frame - track.first_seen < self.confirmation_window:
                stable_id = self.next_stable_id
                self.next_stable_id += 1
                gallery = deque([track.appearance], maxlen=self.gallery_size)
                self.tracks[stable_id] = _StableTrack(stable_id, track.box.copy(), track.appearance, frame,
                                                      track.velocity.copy(), gallery, gallery_frame=frame)
                self.tentative[tentative_index] = None
                ids[valid_indices[local_index]] = stable_id
                self.diagnostics["tracks_confirmed"] += 1
        self.tentative = [track for track in self.tentative if track is not None]

        nuevas = [i for i in remaining if i not in matched_tentative]
        for local_index, signature in zip(nuevas, self._firmas_de(nuevas)):
            if self.min_confirmations == 1:
                stable_id = self.next_stable_id
                self.next_stable_id += 1
                self.tracks[stable_id] = _StableTrack(stable_id, self._cajas[local_index].copy(), signature, frame,
                                                      gallery=deque([signature], maxlen=self.gallery_size),
                                                      gallery_frame=frame)
                ids[valid_indices[local_index]] = stable_id
                self.diagnostics["tracks_confirmed"] += 1
            else:
                self.tentative.append(_TentativeTrack(self._cajas[local_index].copy(), signature, frame, frame))
        return ids

    def resumen(self):
        vistas = max(1, self.diagnostics["detections"])
        return {**self.diagnostics, "appearance_share": round(self.diagnostics["appearance_computed"] / vistas, 3),
                "active_tracks": len(self.tracks), "pending_tracks": len(self.tentative)}

# Parte II — Procesamiento multicámara y mapeo 2D

### 4.1 — Registro técnico de cámaras y homografía

`Modelo/camaras.json` guarda, por cámara, `timestamp_offset`, `resolution`, `H` y `coverage_polygon`. Además registra los solapes entre cámaras y las transiciones $i\rightarrow j$ con $t_{min}\le\Delta t\le t_{max}$.

El apoyo en el suelo es el punto inferior central de la caja, $u=(x_1+x_2)/2,\ v=y_2$, proyectado con $[X',Y',W']^T=H[u,v,1]^T$ y normalizado como $X=X'/W'$, $Y=Y'/W'$. Una homografía solo se acepta si los **puntos de control independientes** tienen un error menor que `max_error_m`.

- `mode = "visual_temporal"`: no hay `H` y la asociación usa apariencia y tiempo. En `trajectory_points`, `x`, `y`, `geom`, `speed_mps` y `direction_deg` quedan `NULL`.
- `mode = "calibrado"`: exige `H` validadas y activa el gating métrico (posición, velocidad y dirección).

In [ ]:
def _puntos(values, minimo=1):
    points = np.asarray(values, dtype=np.float64)
    if points.ndim != 2 or points.shape[1] != 2 or len(points) < minimo or not np.isfinite(points).all():
        raise ValueError(f"Se requieren al menos {minimo} puntos finitos (x, y).")
    return points


def validar_homografia(H):
    matrix = np.asarray(H, dtype=np.float64)
    if matrix.shape != (3, 3) or not np.isfinite(matrix).all() or np.linalg.matrix_rank(matrix) != 3:
        raise ValueError("H debe ser una matriz 3×3 finita e invertible.")
    return matrix / np.linalg.norm(matrix)


def proyectar_puntos(H, puntos):
    matrix, points = validar_homografia(H), _puntos(puntos)
    homogeneous = np.column_stack((points, np.ones(len(points)))) @ matrix.T
    if np.any(np.abs(homogeneous[:, 2]) < 1e-10):
        raise ValueError("Un punto queda en el horizonte de la homografía.")
    return homogeneous[:, :2] / homogeneous[:, 2:3]


def error_reproyeccion(H, imagen, plano):
    image, world = _puntos(imagen), _puntos(plano)
    if image.shape != world.shape:
        raise ValueError("Cada punto de imagen necesita una correspondencia en el plano.")
    errors = np.linalg.norm(proyectar_puntos(H, image) - world, axis=1)
    return {"rmse_m": float(np.sqrt(np.mean(errors ** 2))), "max_error_m": float(errors.max()),
            "errors_m": errors.tolist(), "n": len(errors)}


def calibrar_homografia(imagen, plano, control_imagen, control_plano, *, max_error_m=0.75):
    """Ajusta con ≥ 4 puntos del suelo y valida con ≥ 2 controles que no participan en el ajuste."""
    image, world = _puntos(imagen, 4), _puntos(plano, 4)
    controls, targets = _puntos(control_imagen, 2), _puntos(control_plano, 2)
    if image.shape != world.shape or controls.shape != targets.shape:
        raise ValueError("Las correspondencias tienen longitudes diferentes.")
    if not np.isfinite(max_error_m) or max_error_m <= 0:
        raise ValueError("max_error_m debe ser positivo.")
    for points in (image, world):
        if len(np.unique(points, axis=0)) < 4 or np.linalg.matrix_rank(points - points.mean(0)) < 2:
            raise ValueError("La calibración necesita cuatro puntos distintos no colineales.")
    if any(np.any(np.linalg.norm(image - point, axis=1) < 1e-6) for point in controls):
        raise ValueError("Los puntos de control deben ser independientes del ajuste.")
    H, mask = cv2.findHomography(image, world, cv2.RANSAC, max_error_m)
    if H is None or mask is None or int(mask.sum()) < 4:
        raise ValueError("No se pudo estimar una homografía estable.")
    report = error_reproyeccion(H, controls, targets)
    if report["max_error_m"] > max_error_m:
        raise ValueError(f"Calibración rechazada: error máximo {report['max_error_m']:.3f} m > {max_error_m} m.")
    return {"H": H.tolist(), "fit_image": image.tolist(), "fit_world": world.tolist(),
            "control_image": controls.tolist(), "control_world": targets.tolist(),
            "max_error_m": float(max_error_m), "validation": report}


@dataclass
class Camara:
    camera_id: str
    timestamp_offset: float
    H: np.ndarray | None
    resolution: tuple | None
    coverage_polygon: np.ndarray | None
    validation: dict | None

    def proyectar(self, row, shape, area=None):
        """Punto inferior central de la caja en el plano (metros), o None si no es fiable."""
        if self.H is None:
            return None
        if self.resolution and self.resolution != (shape[1], shape[0]):
            raise ValueError(f"{self.camera_id}: la resolución no coincide con la calibración.")
        if row["y2"] >= (shape[0] if area is None else area[3]) - 2:
            return None  # caja cortada por el borde inferior de la imagen útil: los pies no son visibles
        try:
            point = proyectar_puntos(self.H, [[(row["x1"] + row["x2"]) / 2, row["y2"]]])[0]
        except ValueError:
            return None
        if self.coverage_polygon is not None and cv2.pointPolygonTest(
                self.coverage_polygon, tuple(map(float, point)), False) < 0:
            return None
        return point


def cargar_camaras(config, camera_ids=None):
    data = json.loads(Path(config).read_text()) if isinstance(config, (str, Path)) else config
    mode = data.get("mode", "calibrado")
    if mode not in {"calibrado", "visual_temporal"}:
        raise ValueError("mode debe ser calibrado o visual_temporal.")
    if data.get("units", "m") != "m":
        raise ValueError("El plano común debe utilizar metros (units='m').")
    cameras = {}
    for cid, item in data["cameras"].items():
        offset = float(item.get("timestamp_offset", 0))
        if not np.isfinite(offset):
            raise ValueError(f"{cid}: timestamp_offset no es finito.")
        H = None if item.get("H") is None else validar_homografia(item["H"])
        resolution = tuple(item["resolution"]) if item.get("resolution") else None
        if resolution is not None and (len(resolution) != 2 or any(not isinstance(v, int) or v <= 0 for v in resolution)):
            raise ValueError(f"{cid}: resolution debe ser [ancho, alto].")
        report = None
        if mode == "calibrado":
            if H is None or resolution is None:
                raise ValueError(f"{cid}: faltan H y/o resolution para el modo calibrado.")
            controls = _puntos(item.get("control_image", []), 2)
            targets = _puntos(item.get("control_world", []), 2)
            fit = _puntos(item.get("fit_image", []), 4)
            if any(np.any(np.linalg.norm(fit - p, axis=1) < 1e-6) for p in controls):
                raise ValueError(f"{cid}: los controles repiten puntos del ajuste.")
            report = error_reproyeccion(H, controls, targets)
            limit = float(item.get("max_error_m", 0.75))
            if not np.isfinite(limit) or limit <= 0 or report["max_error_m"] > limit:
                raise ValueError(f"{cid}: la homografía no supera la validación independiente.")
        polygon = _puntos(item["coverage_polygon"], 3).astype(np.float32) if item.get("coverage_polygon") else None
        cameras[cid] = Camara(cid, offset, H, resolution, polygon, report)
    if not cameras or (camera_ids is not None and set(camera_ids) != set(cameras)):
        raise ValueError("Las cámaras del registro deben coincidir con los videos.")
    transitions = {}
    for edge in data.get("transitions", []):
        key = (edge["from"], edge["to"])
        if key[0] not in cameras or key[1] not in cameras or key[0] == key[1] or key in transitions:
            raise ValueError(f"Transición inválida o duplicada: {key}")
        minimum, maximum = float(edge["t_min_s"]), float(edge["t_max_s"])
        distance = float(edge.get("max_distance_m", 30))
        if not all(np.isfinite(x) for x in (minimum, maximum, distance)) or not 0 <= minimum <= maximum or distance <= 0:
            raise ValueError(f"Ventana temporal o distancia inválida: {key}")
        transitions[key] = {**edge, "t_min_s": minimum, "t_max_s": maximum, "max_distance_m": distance}
    overlaps = set()
    for pair in data.get("overlaps", []):
        if len(pair) != 2 or pair[0] == pair[1] or not set(pair) <= set(cameras):
            raise ValueError(f"Solape inválido: {pair}")
        overlaps.add(frozenset(pair))
    return data, cameras, transitions, overlaps

### 4.2 — Tracklets, memoria multivista y Re-ID

Las observaciones de un mismo `local_id` forman un **tracklet** $T=\{camera\_id, local\_id, t_{inicio}, t_{fin}, (X,Y,t), v, \theta, A\}$.

**Encoder:** `yolo26s-reid.onnx` (YOLO26s-ReID entrenado en MSMT17, 512 valores). Usa el mismo preprocesado que Ultralytics (recorte con margen 1.02 + 10 px, RGB/255, resize cuadrado a 448) y se ejecuta **en la GPU**: el ONNX se convierte a PyTorch con `onnx2torch`. Los embeddings son idénticos a los de ONNX Runtime (coseno ≥ 0.99999) y tarda ~5 ms por recorte, frente a ~57 ms en CPU. Si `onnx2torch` no está instalado, usa la CPU.

**Frente, espalda y costado.** Cada tracklet acumula un **prototipo**: el promedio de todas sus vistas, con la orientación anotada (F/E/C/quieto) según el movimiento de los pies en la imagen. Al unirse a una persona, sus vistas pasan a la **galería global compartida**, así que alguien visto de espaldas en `cam01` se reconoce de frente en `cam02`.

Medido con 70 tracklets etiquetados a mano (11 personas, 176 pares positivos entre cámaras):

| Comparación entre cámaras | AUC | Misma persona | Distinta persona |
|---|---|---|---|
| Prototipo Re-ID a 448 | **0.969** | 0.61 | 0.30 |
| Top-3 de muestras sueltas (esquema anterior) | 0.955 | 0.55 | 0.35 |
| Re-ID a 224 | 0.90 | 0.60 | 0.36 |
| Color HSV/Lab por partes | 0.67–0.83 | | |

- **Orientación:** entre frente y espalda la similitud promedió 0.61, igual que en la misma orientación, y una vista nueva se parece a su tracklet 0.69 si cambió la orientación frente a 0.71 si no. El Re-ID ya es casi invariante a la orientación; lo que lo hace robusto es **promediar muchas vistas**. Por eso no se descartan vistas por orientación: guardar solo 4 por orientación bajó el recall de 0.92 a 0.54.
- **Color:** no mejoró el AUC al combinarlo, así que entre cámaras se usa solo el Re-ID. El color sigue en el tracker local.

**Cuándo se pasa una vista por el Re-ID.** Se muestrea una vista fiable cada `sample_interval_s` mientras la persona aún no está bien descrita:
- su tracklet tiene menos de `tracklet_views_target` vistas;
- su ficha en la galería tiene menos de `identity_views_target`;
- o aparece una orientación nueva en ese tracklet.

Después solo se **verifica** cada `verify_interval_s`, salvo que el tracker local avise un cruce o una recuperación, o que haya sospecha de cambio de persona: entonces se revisa en el acto. Así las llamadas al Re-ID bajan 34 %, con precisión 0.99 y recall 0.91 frente a 1.00 y 0.92 muestreando siempre.

**Cambio de persona dentro de un tracklet.** El tracker local a veces entrega el mismo `local_id` a otra persona, por ejemplo en puertas o detrás de columnas. Si `split_strikes` vistas seguidas quedan por debajo de `split_threshold` respecto al prototipo, el tracklet se corta y lo que sigue empieza como identidad nueva. Cuando el `local_id` reaparece tras más de `tracklet_gap_s`, el tramo nuevo tampoco hereda la identidad: decide el Re-ID.

In [ ]:
def _reid_en_torch(weights, device):
    """Convierte el ONNX de Re-ID a PyTorch para ejecutarlo en la GPU (embeddings idénticos: coseno ≥ 0.99999).

    onnx2torch no trae Split/ReduceL2 de opset 18, Reshape/Shape de opset 19 ni GlobalMaxPool; se registran aquí.
    """
    import onnx
    from onnx2torch import convert
    from onnx2torch.node_converters import registry
    from onnx2torch.node_converters.split import OnnxSplit13
    from onnx2torch.utils.common import OnnxToTorchModule, OperationConverterResult, onnx_mapping_from_node

    class ReduceL2Opset18(nn.Module, OnnxToTorchModule):
        def __init__(self, keepdims):
            super().__init__()
            self.keepdims = keepdims

        def forward(self, x, axes=None):
            dim = None if axes is None else [int(a) for a in axes.reshape(-1).tolist()]
            return torch.linalg.vector_norm(x, ord=2, dim=dim, keepdim=self.keepdims)

    class GlobalMaxPoolOnnx(nn.Module, OnnxToTorchModule):
        def forward(self, x):
            return torch.amax(x, dim=tuple(range(2, x.dim())), keepdim=True)

    tabla, clave = registry._CONVERTER_REGISTRY, registry.OperationDescription
    faltantes = {
        ("Split", 18): lambda node, graph: OperationConverterResult(
            torch_module=OnnxSplit13(num_splits=len(node.output_values), axis=node.attributes.get("axis", 0)),
            onnx_mapping=onnx_mapping_from_node(node=node)),
        ("ReduceL2", 18): lambda node, graph: OperationConverterResult(
            torch_module=ReduceL2Opset18(bool(node.attributes.get("keepdims", 1))),
            onnx_mapping=onnx_mapping_from_node(node=node)),
        ("GlobalMaxPool", 1): lambda node, graph: OperationConverterResult(
            torch_module=GlobalMaxPoolOnnx(), onnx_mapping=onnx_mapping_from_node(node=node)),
        ("Reshape", 19): tabla[clave("", "Reshape", 14)],   # opset 19 solo añade tipos float8
        ("Shape", 19): tabla[clave("", "Shape", 15)],
    }
    for (operacion, version), conversor in faltantes.items():
        tabla.setdefault(clave("", operacion, version), conversor)
    modelo = onnx.load(str(weights))
    for nodo in modelo.graph.node:
        while nodo.input and nodo.input[-1] == "":
            nodo.input.pop()  # entrada opcional omitida (Clip sin max)
    return convert(modelo).eval().to(device)


class ReIDPersonas:
    """Encoder de Re-ID corporal yolo26s-reid: en GPU vía PyTorch si es posible; si no, ONNX Runtime en CPU."""

    def __init__(self, weights, imgsz=None, threads=2, batch_size=8, device="auto"):
        import onnxruntime as ort

        if not Path(weights).is_file():
            raise FileNotFoundError(weights)
        if threads < 1 or batch_size < 1:
            raise ValueError("threads y batch_size deben ser positivos.")
        options = ort.SessionOptions()
        options.intra_op_num_threads = threads
        self.session = ort.InferenceSession(str(weights), options, providers=["CPUExecutionProvider"])
        spec = self.session.get_inputs()[0]
        self.input = spec.name
        declared = json.loads(self.session.get_modelmeta().custom_metadata_map.get("imgsz", "[448, 448]"))
        self.imgsz = int(imgsz or declared[0])
        self.height = spec.shape[2] if isinstance(spec.shape[2], int) else self.imgsz
        self.width = spec.shape[3] if isinstance(spec.shape[3], int) else self.imgsz
        self.fixed_batch = spec.shape[0] if isinstance(spec.shape[0], int) else None
        self.batch_size = self.fixed_batch or batch_size
        self.dimension = self.session.get_outputs()[0].shape[-1]
        self.red, self.backend = None, "onnxruntime-cpu"
        if device == "auto":
            device = "cuda:0" if torch.cuda.is_available() else "cpu"
        if str(device).startswith("cuda"):
            try:
                self.red, self.device = _reid_en_torch(weights, device), device
                self.backend = f"pytorch-{device}"
            except Exception as exc:  # sin onnx2torch o grafo no convertible: se mantiene la CPU
                print(f"Re-ID en CPU (no se pudo usar la GPU: {type(exc).__name__}: {exc})")

    def __call__(self, crops):
        if not crops:
            return np.empty((0, self.dimension), np.float32)
        if any(not crop.size for crop in crops):
            raise ValueError("El encoder recibió un recorte vacío.")
        outputs = []
        for start in range(0, len(crops), self.batch_size):
            chunk = crops[start:start + self.batch_size]
            batch = np.stack([cv2.resize(crop, (self.width, self.height), interpolation=cv2.INTER_LINEAR)
                              for crop in chunk])[..., ::-1]
            tensor = np.ascontiguousarray(batch.transpose(0, 3, 1, 2), dtype=np.float32) / 255.0
            if self.red is not None:
                with torch.inference_mode():
                    outputs.append(self.red(torch.from_numpy(tensor).to(self.device)).float().cpu().numpy())
                continue
            if self.fixed_batch and len(tensor) < self.fixed_batch:
                tensor = np.concatenate((tensor, np.repeat(tensor[-1:], self.fixed_batch - len(tensor), axis=0)))
            outputs.append(self.session.run(None, {self.input: tensor})[0][:len(chunk)])
        features = np.concatenate(outputs)
        if features.ndim != 2 or not np.isfinite(features).all():
            raise ValueError("El encoder produjo descriptores inválidos.")
        return features / (np.linalg.norm(features, axis=1, keepdims=True) + 1e-12)


class ReIDResNet:
    """Alternativa sin pesos extra: reutiliza la ResNet18 de la Parte I."""

    def __init__(self, extractor):
        self.extractor = extractor

    def __call__(self, crops):
        return np.stack([self.extractor(crop, np.array([[0, 0, crop.shape[1], crop.shape[0]]], np.float32))[0]
                         for crop in crops])


def recortar(image, row):
    """Recorte con el mismo margen que Ultralytics (save_one_box: gain 1.02, pad 10)."""
    cx, cy = (row["x1"] + row["x2"]) / 2, (row["y1"] + row["y2"]) / 2
    w, h = (row["x2"] - row["x1"]) * 1.02 + 10, (row["y2"] - row["y1"]) * 1.02 + 10
    height, width = image.shape[:2]
    return image[max(0, int(cy - h / 2)):min(height, int(cy + h / 2)),
                 max(0, int(cx - w / 2)):min(width, int(cx + w / 2))]


def vistas_confiables(rows, shape, min_conf=0.55, min_alto=40, max_iou=0.2, margen_borde=4, occluders=None):
    """Índices de filas aptas para Re-ID: buena confianza, altura suficiente, lejos del borde y sin oclusión.

    Las detecciones aún tentativas también cuentan como oclusores.
    """
    x0, y0, width, height = 0, 0, shape[1], shape[0]
    boxes = np.array([[r[k] for k in ("x1", "y1", "x2", "y2")] for r in rows], np.float32).reshape(-1, 4)
    others = boxes if occluders is None else np.asarray(occluders, dtype=np.float32).reshape(-1, 4)
    keep = []
    for index, (row, box) in enumerate(zip(rows, boxes)):
        if row["confidence"] < min_conf or box[3] - box[1] < min_alto or box[2] - box[0] < 8:
            continue
        if (box[0] < x0 + margen_borde or box[1] < y0 + margen_borde
                or box[2] > width - margen_borde or box[3] > height - margen_borde):
            continue
        covered = False
        for other in others:
            if np.allclose(box, other, atol=1):
                continue
            intersection = np.prod(np.maximum(0, np.minimum(box[2:], other[2:]) - np.maximum(box[:2], other[:2])))
            if _iou(box, other) > max_iou or intersection / max(1, np.prod(box[2:] - box[:2])) > max_iou:
                covered = True
                break
        if not covered:
            keep.append(index)
    return keep


@dataclass
class Tracklet:
    uid: str                 # etiqueta legible: cam01/L3/T1
    tracklet_uuid: str       # UUID de trajectory_points.tracklet_id
    camera_id: str
    local_id: int
    inicio_s: float
    fin_s: float
    global_id: int
    vistas: dict = field(default_factory=dict)          # orientación (F/E/C/?) -> número de vistas guardadas
    suma: np.ndarray | None = None                      # suma de embeddings Re-ID normalizados
    n_muestras: int = 0
    pendientes: list = field(default_factory=list)      # vistas seguidas que no encajan (posible cambio de persona)
    verificar: bool = False                             # el tracker local avisó un cruce o recuperación
    huella: deque = field(default_factory=lambda: deque(maxlen=24))   # (t, x centro, y pies, alto) en la imagen
    tiempos: deque = field(default_factory=lambda: deque(maxlen=120))
    posiciones: deque = field(default_factory=lambda: deque(maxlen=180))
    primera_posicion: tuple | None = None
    ultima_posicion: tuple | None = None
    ultimo_muestreo: float = -math.inf
    observaciones: int = 0
    match_score: float | None = None
    corte_s: float = math.inf        # desde este instante las filas pertenecen a `sucesor`
    sucesor: str | None = None

    def prototipo(self):
        """Memoria multivista: promedio de todas las vistas (frente, espalda y costado)."""
        return None if self.suma is None else self.suma / (np.linalg.norm(self.suma) + 1e-12)

    def orientacion(self):
        """F/E/C según el desplazamiento de los pies en la imagen (cámara elevada); '?' si está quieto."""
        if len(self.huella) < 2:
            return "?"
        ultimo = self.huella[-1]
        primero = next((p for p in self.huella if ultimo[0] - p[0] <= 0.8), None)
        dt = ultimo[0] - primero[0]
        if dt < 0.3:
            return "?"
        alto = max(ultimo[3], 1.0)
        dx, dy = (ultimo[1] - primero[1]) / alto / dt, (ultimo[2] - primero[2]) / alto / dt
        if math.hypot(dx, dy) < 0.15:
            return "?"
        if abs(dy) >= 0.7 * abs(dx):
            return "F" if dy > 0 else "E"
        return "C"

    def guardar_vista(self, orientacion, vector):
        self.vistas[orientacion] = self.vistas.get(orientacion, 0) + 1
        self.suma = vector.copy() if self.suma is None else self.suma + vector
        self.n_muestras += 1

    def movimiento(self):
        """Velocidad (m/s) en el plano con una ventana ≥ 0.25 s para no amplificar el jitter."""
        if len(self.posiciones) < 2:
            return None
        last = self.posiciones[-1]
        first = next((p for p in reversed(self.posiciones) if 0.25 <= last[0] - p[0] <= 1.5), None)
        if first is None:
            return None
        return (np.array(last[1:]) - first[1:]) / (last[0] - first[0])


def _solapan(a, b):
    return min(a.fin_s, b.fin_s) - max(a.inicio_s, b.inicio_s) >= -1e-8

### 4.3 — Galería global compartida y `global_id`

Todas las cámaras se analizan **a la vez contra la misma memoria**. Cada persona es una ficha de la galería: las vistas Re-ID de todos sus tracklets en las tres cámaras (con su orientación), dónde y cuándo se vio por última vez, y cuánto tiempo lleva visible. Cuando una cámara ve a alguien, lo compara con esa ficha, que ya incluye lo que vieron las otras cámaras antes.

**Identidades provisionales y pasadas breves.** Una persona nueva empieza sin ID público:
- **Se une a alguien conocido** si con `min_query_samples` vistas ya coincide con una ficha confirmada.
- **Recibe un ID nuevo** solo si estuvo visible `min_identity_duration_s` y reunió `min_samples` vistas.
- **No se cuenta** si pasó poco tiempo por la cámara y no coincide con nadie: no aparece en `trajectory_points` ni en el total de personas.

Los IDs públicos son consecutivos (1..N) por orden de aparición.

Dos identidades se fusionan cuando:

1. **Son físicamente posibles.** Una identidad no puede tener dos tracklets simultáneos en la misma cámara. Dos cámaras solo comparten a una persona al mismo tiempo si están declaradas como solapadas. Entre segmentos consecutivos sin solape debe existir la transición $i\rightarrow j$ dentro de $[t_{min}, t_{max}]$, y una re-entrada en la misma cámara debe ocurrir dentro de `reentry_window_s`. En modo calibrado se exige además coherencia de posición, dirección y velocidad.
2. **Se parecen lo suficiente.** El coseno entre las fichas debe superar `threshold` (0.60), y el **enlace promedio** entre sus tracklets debe superar `average_threshold` (0.55). Esto evita que un único tracklet parecido encadene a personas distintas.
3. **La re-entrada en una sola cámara exige más.** Si toda la evidencia viene de una misma cámara, se usa `same_camera_threshold` (0.70): con la misma luz y el mismo fondo, personas distintas se parecen más (0.45 de media, frente a 0.30 entre cámaras).
4. **No hay ambigüedad.** Si la mejor candidata y otra que no puede ser la misma persona quedan a menos de `ambiguity_margin`, se espera. El emparejamiento es **mutuo**: la ficha destino también debe preferir a quien la propone.
5. **Se fusiona con la identidad más antigua**, que conserva su ID público. Si una fusión deja de ser físicamente posible, el tracklet en conflicto se separa.

**Resultado** con los videos completos y 70 tracklets etiquetados, sobre pares de tracklets:

| Configuración | Precisión | Recall | Personas contadas |
|---|---|---|---|
| Esquema anterior (top-k + color, umbral 0.78) | — | — | 1 fusión en 94 tracklets |
| **Actual, corrida real en GPU** | **0.99** | **0.86** | 20 (+21 pasadas breves no contadas) |

Estos números son de este dataset: con otras cámaras conviene volver a medir. Los `global_id` dibujados en vivo son decisiones *online*; la tabla final y los videos exportados usan la identidad **reconciliada** al terminar.

In [ ]:
class AsociadorMulticamara:
    def __init__(self, reid, config):
        self.config, self.camaras, self.transiciones, self.solapes = cargar_camaras(config)
        settings = self.config.get("association", {})
        self.mode = self.config.get("mode", "calibrado")
        self.reid = reid
        self.sample_s = float(settings.get("sample_interval_s", 0.4))
        self.tracklet_views_target = int(settings.get("tracklet_views_target", 8))
        self.identity_views_target = int(settings.get("identity_views_target", 30))
        self.verify_s = float(settings.get("verify_interval_s", 2.0))
        self.min_samples = int(settings.get("min_samples", 5))
        self.min_query_samples = int(settings.get("min_query_samples", 3))
        self.min_duration_s = float(settings.get("min_identity_duration_s", 2.0))
        self.threshold = float(settings.get("threshold", 0.60))
        self.average_threshold = float(settings.get("average_threshold", 0.55))
        self.same_camera_threshold = float(settings.get("same_camera_threshold", 0.70))
        self.margin = float(settings.get("ambiguity_margin", 0.05))
        self.mutual = bool(settings.get("mutual_best", True))
        self.split_threshold = float(settings.get("split_threshold", 0.40))
        self.split_strikes = int(settings.get("split_strikes", 2))
        self.window = float(settings.get("candidate_window_s", 90))
        self.reentry = float(settings.get("reentry_window_s", 120))
        self.gap = float(settings.get("tracklet_gap_s", 2))
        self.tolerance = float(settings.get("sync_tolerance_s", 0.15))
        self.max_distance = float(settings.get("overlap_distance_m", 1.5))
        self.max_speed = float(settings.get("max_speed_m_s", 4))
        self.quality = dict(min_conf=float(settings.get("min_conf", 0.55)),
                            min_alto=float(settings.get("min_height", 40)),
                            max_iou=float(settings.get("max_occlusion", 0.2)), margen_borde=4)
        if not 1 <= self.min_query_samples <= self.min_samples or self.split_strikes < 1 or self.tracklet_views_target < 1 or self.identity_views_target < 1 or self.min_duration_s < 0 or any(
                not np.isfinite(v) or v <= 0 for v in (self.sample_s, self.verify_s, self.window,
                                                       self.reentry, self.gap, self.tolerance, self.max_distance, self.max_speed)):
            raise ValueError("Parámetros temporales, de muestras o físicos inválidos.")
        if any(not np.isfinite(v) or not -1 <= v <= 1 for v in (
                self.threshold, self.average_threshold, self.same_camera_threshold, self.margin, self.split_threshold)):
            raise ValueError("Los umbrales de similitud deben estar entre -1 y 1.")
        self.areas = {}  # camera_id -> (x0, y0, x1, y1) con imagen real
        self.reiniciar()

    # ------------------------------------------------------------------ estado
    def reiniciar(self, session_uuid=None):
        self.session_uuid = session_uuid or uuid.uuid4()
        self.tracklets = {}          # uid -> Tracklet
        self.locales = {}            # (camera_id, local_id) -> Tracklet vigente
        self.globales = {}           # identidad interna -> {uid}: la galería global compartida por todas las cámaras
        self.confirmadas = {}        # identidad interna -> ID público (solo personas contadas)
        self.siguiente_publico = 1
        self.segmentos = {}
        self.siguiente = 1
        self.vigentes = set()        # identidades que aún pueden recibir fusiones
        self.sucias = set()          # identidades con vistas nuevas en este instante
        self.last_camera_time = {}
        self.last_timestamp = -math.inf
        self.evidencia_solapes = {}
        self.stats = {"muestras_reid": 0, "verificaciones": 0,
                      "fusiones": 0, "separaciones": 0, "cambios_de_persona": 0, "ambiguos": 0}
        self.enlaces = []

    def global_uuid(self, publico):
        return str(uuid.uuid5(self.session_uuid, f"G{publico}"))

    def _nuevo(self, cid, lid, timestamp):
        key = (cid, lid)
        self.segmentos[key] = self.segmentos.get(key, 0) + 1
        uid = f"{cid}/L{lid}/T{self.segmentos[key]}"
        track = Tracklet(uid, str(uuid.uuid5(self.session_uuid, uid)), cid, lid, timestamp, timestamp, self.siguiente)
        self.globales[self.siguiente] = {uid}
        self.vigentes.add(self.siguiente)
        self.siguiente += 1
        self.tracklets[uid] = self.locales[key] = track
        return track

    def resolver(self, uid, timestamp):
        """Tracklet real de una fila: sigue los cortes hechos cuando el Re-ID detectó otra persona."""
        track = self.tracklets[uid]
        while track.sucesor is not None and timestamp >= track.corte_s - 1e-9:
            track = self.tracklets[track.sucesor]
        return track

    def _miembros(self, gid):
        return [self.tracklets[uid] for uid in self.globales[gid]]

    def _n_vistas(self, gid):
        return sum(self.tracklets[uid].n_muestras for uid in self.globales[gid])

    def _minimo(self, gid, otra):
        """Vistas necesarias: una pasada provisional puede consultar a una identidad ya confirmada con menos vistas."""
        return self.min_query_samples if gid not in self.confirmadas and otra in self.confirmadas else self.min_samples

    def _duracion_visible(self, gid):
        """Segundos en que la identidad estuvo visible en alguna cámara (unión de intervalos)."""
        intervalos = sorted((t.inicio_s, t.fin_s) for t in self._miembros(gid))
        total, (inicio, fin) = 0.0, intervalos[0]
        for a, b in intervalos[1:]:
            if a > fin:
                total, inicio, fin = total + fin - inicio, a, b
            else:
                fin = max(fin, b)
        return total + fin - inicio

    def _fin(self, gid):
        return max(self.tracklets[uid].fin_s for uid in self.globales[gid])

    # ------------------------------------------------------------ observaciones
    def actualizar(self, timestamp, frames, observaciones, occluders=None):
        """Recibe las filas locales de todas las cámaras de un mismo instante y les asigna global_id."""
        if not math.isfinite(timestamp) or timestamp < self.last_timestamp - 1e-8:
            raise ValueError("El asociador necesita frames ordenados por timestamp corregido.")
        self.last_timestamp = timestamp
        crops, sampled, tocados = [], [], set()
        for cid, rows in observaciones.items():
            if cid not in self.camaras or cid not in frames:
                raise ValueError(f"Cámara sin registro o sin frame: {cid}")
            if timestamp <= self.last_camera_time.get(cid, -math.inf):
                raise ValueError(f"{cid}: timestamp repetido o fuera de orden.")
            self.last_camera_time[cid] = timestamp
            seen = set()
            for row in rows:
                lid = int(row["local_id"])
                if lid in seen:
                    raise ValueError(f"{cid}: local_id duplicado en un frame.")
                seen.add(lid)
                track = self.locales.get((cid, lid))
                if track is None or timestamp - track.fin_s > self.gap:
                    # Tras una pausa el tracker local puede reutilizar el ID con otra persona:
                    # el tramo nuevo empieza como identidad propia y el Re-ID decide si se une.
                    track = self._nuevo(cid, lid, timestamp)
                track.fin_s = timestamp
                track.tiempos.append(timestamp)
                track.huella.append((timestamp, (row["x1"] + row["x2"]) / 2, row["y2"], row["y2"] - row["y1"]))
                track.verificar |= bool(row.get("riesgo"))
                track.observaciones += 1
                row["tracklet_id"] = track.uid
                tocados.add((cid, lid))
                self._proyectar(track, row, timestamp, frames[cid].shape)
            obstacles = None if occluders is None else occluders.get(cid)
            for index in vistas_confiables(rows, frames[cid].shape, **self.quality, occluders=obstacles):
                row = rows[index]
                track = self.tracklets[row["tracklet_id"]]
                if self._debe_muestrear(track, row, timestamp):
                    crop = recortar(frames[cid], row)
                    if crop.size:
                        track.ultimo_muestreo = timestamp
                        crops.append(crop)
                        sampled.append((track, track.orientacion()))
        if crops:
            vectors = self.reid(crops)
            if len(vectors) != len(sampled):
                raise ValueError("El número de embeddings no coincide con los recortes.")
            for (track, orientacion), vector in zip(sampled, vectors):
                vector = np.asarray(vector, dtype=np.float32).reshape(-1)
                norm = np.linalg.norm(vector)
                if np.isfinite(vector).all() and norm > 1e-8:
                    self.stats["muestras_reid"] += 1
                    self._agregar_vista(self.locales[(track.camera_id, track.local_id)], vector / norm, timestamp,
                                        orientacion)
        self._separar_conflictos({self.locales[key].global_id for key in tocados})
        if self.sucias:
            self._asociar()
        self._confirmar({self.locales[key].global_id for key in tocados})
        for rows in observaciones.values():
            for row in rows:
                track = self.resolver(row["tracklet_id"], timestamp)
                row["tracklet_id"], row["global_id"] = track.uid, self.confirmadas.get(track.global_id)

    def _confirmar(self, gids):
        """Una identidad nueva solo se cuenta si estuvo visible `min_identity_duration_s` y reunió `min_samples` vistas.

        Antes de eso es provisional: puede unirse a una persona conocida, pero no recibe un ID público propio.
        """
        for gid in sorted(gids):
            if (gid in self.globales and gid not in self.confirmadas and self._n_vistas(gid) >= self.min_samples
                    and self._duracion_visible(gid) >= self.min_duration_s):
                self.confirmadas[gid] = self.siguiente_publico
                self.siguiente_publico += 1

    def _debe_muestrear(self, track, row, timestamp):
        """Decide si vale la pena pasar esta vista por el Re-ID, la parte más costosa del pipeline.

        Se muestrea a ritmo normal mientras la persona aún no está bien descrita: su tracklet tiene menos de
        `tracklet_views_target` vistas, su ficha en la galería tiene menos de `identity_views_target` o aparece
        una orientación que este tracklet aún no tenía. Una vez descrita, solo se verifica cada `verify_interval_s`,
        salvo que el tracker local avise un cruce o haya sospecha de cambio de persona (entonces se revisa ya).
        """
        if timestamp - track.ultimo_muestreo + 1e-8 < self.sample_s:
            return False
        if track.pendientes or track.verificar:
            return True
        if (track.n_muestras < self.tracklet_views_target or not track.vistas.get(track.orientacion())
                or self._n_vistas(track.global_id) < self.identity_views_target):
            return True
        return timestamp - track.ultimo_muestreo + 1e-8 >= self.verify_s

    def _agregar_vista(self, track, vector, timestamp, orientacion):
        """Verifica la vista contra el prototipo; si encaja se suma, si varias seguidas no encajan se corta el tracklet."""
        prototipo = track.prototipo()
        if prototipo is not None and track.n_muestras >= 3 and float(vector @ prototipo) < self.split_threshold:
            track.pendientes.append((timestamp, vector, orientacion))
            if len(track.pendientes) >= self.split_strikes:
                self._partir(track)
            return
        if track.verificar or (track.n_muestras >= self.tracklet_views_target
                               and self._n_vistas(track.global_id) >= self.identity_views_target):
            self.stats["verificaciones"] += 1
        track.pendientes.clear()  # una vista atípica aislada (oclusión, pose extrema) se descarta
        track.verificar = False
        track.guardar_vista(orientacion, vector)
        self.sucias.add(track.global_id)

    def _partir(self, track):
        corte = track.pendientes[0][0]
        nuevo = self._nuevo(track.camera_id, track.local_id, corte)
        nuevo.fin_s, nuevo.ultimo_muestreo = track.fin_s, track.ultimo_muestreo
        nuevo.tiempos.extend(t for t in track.tiempos if t >= corte - 1e-9)
        nuevo.huella.extend(h for h in track.huella if h[0] >= corte - 1e-9)
        nuevo.observaciones = len(nuevo.tiempos)
        for _, vector, orientacion in track.pendientes:
            nuevo.guardar_vista(orientacion, vector)
        anteriores = [t for t in track.tiempos if t < corte - 1e-9]
        track.fin_s = anteriores[-1] if anteriores else track.inicio_s
        track.observaciones = max(0, track.observaciones - nuevo.observaciones)
        track.corte_s, track.sucesor = corte, nuevo.uid
        track.pendientes.clear()
        nuevo.posiciones.extend(p for p in track.posiciones if p[0] >= corte - 1e-9)
        track.posiciones = deque((p for p in track.posiciones if p[0] < corte - 1e-9), maxlen=track.posiciones.maxlen)
        for t in (track, nuevo):
            t.primera_posicion = t.posiciones[0] if t.posiciones else None
            t.ultima_posicion = t.posiciones[-1] if t.posiciones else None
        self.stats["cambios_de_persona"] += 1
        self.sucias.add(nuevo.global_id)
        self.enlaces.append({"timestamp_s": corte, "event": "split", "global_id": nuevo.global_id,
                             "absorbed_global_id": track.global_id, "score": None, "average": None,
                             "tracklet_a": track.uid, "tracklet_b": nuevo.uid})

    def _proyectar(self, track, row, timestamp, shape):
        row.update(X=None, Y=None, speed=None, direction_deg=None, projection_valid=False)
        point = self.camaras[track.camera_id].proyectar(row, shape, self.areas.get(track.camera_id))
        if point is None:
            return
        current = (timestamp, float(point[0]), float(point[1]))
        if track.ultima_posicion is not None:
            previous = track.ultima_posicion
            if math.dist(current[1:], previous[1:]) > self.max_speed * (timestamp - previous[0]) + self.max_distance:
                return  # salto físicamente imposible: se descarta esta proyección
        track.posiciones.append(current)
        track.primera_posicion = track.primera_posicion or current
        track.ultima_posicion = current
        row.update(X=current[1], Y=current[2], projection_valid=True)
        velocity = track.movimiento()
        if velocity is not None:
            row["speed"] = float(np.hypot(velocity[0], velocity[1]))
            if row["speed"] > 0.05:
                row["direction_deg"] = float(np.degrees(np.arctan2(velocity[1], velocity[0])) % 360.0)

    # ------------------------------------------------------------ gating físico
    def _fisica(self, a, b):
        """None si a y b no pueden ser la misma persona; si pueden, términos time/pos/dir/vel en [0, 1]."""
        if a.camera_id == b.camera_id:
            if _solapan(a, b):
                return None  # dos tracklets simultáneos en una cámara son dos personas
            gap = max(a.inicio_s, b.inicio_s) - min(a.fin_s, b.fin_s)
            return None if gap > self.reentry else {"time": 1.0}
        if _solapan(a, b):
            if frozenset((a.camera_id, b.camera_id)) not in self.solapes:
                return None
            terms = {"time": 1.0}
            if self.mode == "calibrado":
                key = tuple(sorted((a.uid, b.uid)))
                if not a.posiciones or not b.posiciones:
                    return self.evidencia_solapes.get(key)
                pairs = []
                for p in list(a.posiciones)[-30:]:
                    q = min(b.posiciones, key=lambda value: abs(value[0] - p[0]))
                    if abs(q[0] - p[0]) <= self.tolerance:
                        pairs.append((math.dist(p[1:], q[1:]), self.max_distance + self.max_speed * abs(q[0] - p[0])))
                if not pairs:
                    return self.evidencia_solapes.get(key)
                if any(distance > limit for distance, limit in pairs):
                    self.evidencia_solapes.pop(key, None)
                    return None
                terms["pos"] = max(0.0, 1 - float(np.mean([d / limit for d, limit in pairs])))
                self.evidencia_solapes[key] = terms.copy()
            return self._movimiento(a, b, terms)

        source, target = (a, b) if a.inicio_s <= b.inicio_s else (b, a)
        edge = self.transiciones.get((source.camera_id, target.camera_id))
        dt = target.inicio_s - source.fin_s
        if edge is None or not edge["t_min_s"] <= dt <= edge["t_max_s"]:
            return None
        width = max(edge["t_max_s"] - edge["t_min_s"], 1e-6) / 2
        terms = {"time": float(math.exp(-0.5 * ((dt - edge["t_min_s"]) / width) ** 2))}
        if self.mode == "calibrado":
            p, q = source.ultima_posicion, target.primera_posicion
            if p is None or q is None or abs(source.fin_s - p[0]) > self.gap or abs(q[0] - target.inicio_s) > self.gap:
                return None
            distance = math.dist(p[1:], q[1:])
            limit = min(edge["max_distance_m"], self.max_speed * dt + self.max_distance)
            if distance > limit:
                return None
            velocity = source.movimiento()
            if velocity is not None and np.linalg.norm(velocity) > 0.3 and distance > self.max_distance:
                delta = np.array(q[1:]) - p[1:]
                if float(velocity @ delta / (np.linalg.norm(velocity) * distance)) < float(edge.get("min_direction_cos", -0.5)):
                    return None
            terms["pos"] = max(0.0, 1 - distance / max(limit, 1e-8))
        return self._movimiento(a, b, terms)

    def _movimiento(self, a, b, terms):
        if self.mode != "calibrado":
            return terms
        va, vb = a.movimiento(), b.movimiento()
        if va is not None and vb is not None:
            sa, sb = float(np.linalg.norm(va)), float(np.linalg.norm(vb))
            if max(sa, sb) > self.max_speed:
                return None
            terms["vel"] = math.exp(-abs(sa - sb) / self.max_speed)
            if min(sa, sb) > 0.3:
                terms["dir"] = float((np.clip(va @ vb / (sa * sb), -1, 1) + 1) / 2)
        return terms

    def _compatibles(self, grupo_a, grupo_b):
        """¿Pueden dos identidades ser la misma persona? Solo revisa pares nuevos (a ∈ A, b ∈ B)."""
        for a in grupo_a:
            for b in grupo_b:
                if (_solapan(a, b) or a.camera_id == b.camera_id) and self._fisica(a, b) is None:
                    return False
        origen = {t.uid: 0 for t in grupo_a} | {t.uid: 1 for t in grupo_b}
        union = sorted(grupo_a + grupo_b, key=lambda t: (t.inicio_s, t.uid))
        for i, member in enumerate(union):
            if not i or any(o.fin_s >= member.inicio_s - 1e-8 for o in union[:i]):
                continue  # la persona ya estaba visible cuando empezó este tracklet
            previous = max(union[:i], key=lambda t: t.fin_s)
            if origen[previous.uid] != origen[member.uid] and self._fisica(previous, member) is None:
                return False  # transición sin arista o fuera de su ventana temporal
        return True

    # -------------------------------------------------------------- similitud
    def _similitud(self, g1, g2):
        """(centroide, promedio, umbral): prototipo multivista de cada identidad, enlace promedio y umbral aplicable."""
        if self._n_vistas(g1) < self._minimo(g1, g2) or self._n_vistas(g2) < self._minimo(g2, g1):
            return None
        grupo_a, grupo_b = self._con_vistas(g1), self._con_vistas(g2)
        # Solo evidencia de una misma cámara (re-entrada): misma luz y fondo, se exige más parecido.
        una_camara = len({t.camera_id for t in grupo_a + grupo_b}) == 1
        umbral = self.same_camera_threshold if una_camara else self.threshold
        suma_a, suma_b = sum(t.suma for t in grupo_a), sum(t.suma for t in grupo_b)
        centroide = float(suma_a @ suma_b / (np.linalg.norm(suma_a) * np.linalg.norm(suma_b) + 1e-12))
        promedio = float((np.stack([t.prototipo() for t in grupo_a]) @ np.stack([t.prototipo() for t in grupo_b]).T).mean())
        return centroide, promedio, umbral

    def _con_vistas(self, gid):
        """Tracklets que aportan al enlace promedio: los que tienen vistas suficientes o, si no hay, todos los que tienen alguna."""
        miembros = [t for t in self._miembros(gid) if t.n_muestras]
        solidos = [t for t in miembros if t.n_muestras >= self.min_query_samples]
        return solidos or miembros

    # --------------------------------------------------------------- fusiones
    def _asociar(self):
        sucias, self.sucias = self.sucias, set()
        self.vigentes = {g for g in self.vigentes if g in self.globales and self._fin(g) >= self.last_timestamp - self.window}
        listas = [g for g in sorted(self.vigentes) if self._n_vistas(g) >= self.min_query_samples]
        propuestas = []
        for gb in sorted(g for g in sucias if g in self.globales and self._n_vistas(g) >= self.min_query_samples):
            candidatas = []
            for ga in listas:
                if ga == gb:
                    continue
                similitud = self._similitud(ga, gb)
                if (similitud is None or similitud[0] < similitud[2]
                        or similitud[1] < self.average_threshold + similitud[2] - self.threshold):
                    continue
                if self._compatibles(self._miembros(ga), self._miembros(gb)):
                    candidatas.append((similitud[0], ga, similitud[1]))
            if not candidatas:
                continue
            candidatas.sort(key=lambda c: -c[0])
            best = candidatas[0]
            # Empate con otra candidata que no puede ser la misma persona: no se decide todavía.
            if any(best[0] - c[0] < self.margin and not self._compatibles(self._miembros(best[1]), self._miembros(c[1]))
                   for c in candidatas[1:]):
                self.stats["ambiguos"] += 1
                continue
            propuestas.append((best[0], gb, best[1], best[2]))
        if self.mutual:
            # Emparejamiento mutuo: la identidad destino también debe preferir a quien la propone.
            mejor_de = {}
            for propuesta in propuestas:
                if propuesta[0] > mejor_de.get(propuesta[2], (-2.0,))[0]:
                    mejor_de[propuesta[2]] = propuesta
            propuestas = [p for p in propuestas if mejor_de[p[2]] is p]
        usadas = set()
        for score, gb, ga, promedio in sorted(propuestas, key=lambda p: -p[0]):
            if gb not in usadas and ga not in usadas and self._fusionar(ga, gb, score, promedio):
                usadas.update((ga, gb))

    def _fusionar(self, g1, g2, score, promedio):
        if g1 not in self.globales or g2 not in self.globales or g1 == g2:
            return False
        destino, origen = min(g1, g2), max(g1, g2)  # se conserva el GID más antiguo
        if not self._compatibles(self._miembros(destino), self._miembros(origen)):
            return False
        absorbidos = sorted(self.globales[origen])
        publicos = [self.confirmadas.pop(g) for g in (destino, origen) if g in self.confirmadas]
        if publicos:
            self.confirmadas[destino] = min(publicos)  # la persona conserva su ID público más antiguo
        for uid in self.globales[origen] | self.globales[destino]:
            track = self.tracklets[uid]
            track.match_score = max(score, track.match_score or -1.0)
            track.global_id = destino
        self.globales[destino] |= self.globales.pop(origen)
        self.vigentes.discard(origen)
        self.vigentes.add(destino)
        self.stats["fusiones"] += 1
        self.enlaces.append({"timestamp_s": self.last_timestamp, "event": "associate", "global_id": destino,
                             "absorbed_global_id": origen, "score": float(score), "average": float(promedio),
                             "tracklet_a": ", ".join(absorbidos), "tracklet_b": None})
        return True

    def _separar_conflictos(self, gids):
        for gid in sorted(gids):
            uids = self.globales.get(gid)
            if not uids or len(uids) < 2:
                continue
            aceptados = []
            for track in sorted((self.tracklets[u] for u in uids), key=lambda t: (t.inicio_s, t.uid)):
                if not any(_solapan(track, other) and self._fisica(track, other) is None for other in aceptados):
                    aceptados.append(track)
                    continue
                uids.remove(track.uid)
                track.global_id, track.match_score = self.siguiente, None
                self.globales[self.siguiente] = {track.uid}
                self.vigentes.add(self.siguiente)
                self.sucias.add(self.siguiente)
                self.siguiente += 1
                self.stats["separaciones"] += 1
                self.enlaces.append({"timestamp_s": self.last_timestamp, "event": "separate",
                                     "global_id": track.global_id, "absorbed_global_id": gid, "score": None,
                                     "average": None, "tracklet_a": track.uid, "tracklet_b": None})

    # -------------------------------------------------------------- reportes
    def numeracion_final(self):
        """IDs públicos consecutivos (1..N) por orden de primera aparición de cada persona contada."""
        orden = sorted(self.confirmadas, key=lambda g: (min(self.tracklets[u].inicio_s for u in self.globales[g]), g))
        return {gid: i + 1 for i, gid in enumerate(orden)}

    def resumen(self):
        return {"mode": self.mode, "geometria_validada": self.mode == "calibrado",
                "identidades_globales": len(self.confirmadas),
                "pasadas_breves_no_contadas": len(self.globales) - len(self.confirmadas),
                "tracklets": len(self.tracklets),
                "identidades_multicamara": sum(len({self.tracklets[u].camera_id for u in self.globales[g]}) > 1
                                               for g in self.confirmadas), **self.stats}

    def galeria(self, numeracion=None):
        """Metadata compartida por todas las cámaras: qué se sabe de cada persona contada."""
        numeracion = self.confirmadas if numeracion is None else numeracion
        filas = []
        for gid, publico in sorted(numeracion.items(), key=lambda kv: kv[1]):
            miembros = self._miembros(gid)
            orientaciones = {o: 0 for o in ("F", "E", "C", "?")}
            por_camara = {}
            for t in miembros:
                por_camara[t.camera_id] = por_camara.get(t.camera_id, 0) + t.n_muestras
                for o, cantidad in t.vistas.items():
                    orientaciones[o] += cantidad
            ultimo = max(miembros, key=lambda t: t.fin_s)
            filas.append({"global_id": publico, "vistas": sum(por_camara.values()), "vistas_por_camara": por_camara,
                          "frente": orientaciones["F"], "espalda": orientaciones["E"], "costado": orientaciones["C"],
                          "quieto": orientaciones["?"], "segundos_visible": round(self._duracion_visible(gid), 1),
                          "ultima_camara": ultimo.camera_id, "ultima_vez_s": round(ultimo.fin_s, 2)})
        return filas

    def identidades(self, numeracion=None):
        numeracion = self.confirmadas if numeracion is None else numeracion
        return [{"global_id": publico, "global_uuid": self.global_uuid(publico),
                 "camaras": sorted({self.tracklets[u].camera_id for u in self.globales[gid]}),
                 "tracklets": sorted(self.globales[gid]),
                 "inicio_s": min(self.tracklets[u].inicio_s for u in self.globales[gid]),
                 "fin_s": max(self.tracklets[u].fin_s for u in self.globales[gid])}
                for gid, publico in sorted(numeracion.items(), key=lambda kv: kv[1])]

    def tabla_tracklets(self, numeracion=None):
        numeracion = self.confirmadas if numeracion is None else numeracion
        return [{"tracklet_id": t.tracklet_uuid, "tracklet": t.uid, "camera_id": t.camera_id, "local_id": t.local_id,
                 "global_id": numeracion.get(t.global_id), "contado": t.global_id in numeracion,
                 "inicio_s": t.inicio_s, "fin_s": t.fin_s, "observations": t.observaciones, "samples": t.n_muestras,
                 "frente": t.vistas.get("F", 0), "espalda": t.vistas.get("E", 0),
                 "costado": t.vistas.get("C", 0), "entry": t.primera_posicion, "exit": t.ultima_posicion,
                 "association_score": t.match_score}
                for t in self.tracklets.values()]

## 5. Motor de procesamiento

`MotorLAP01` carga una sola vez YOLO26m y ResNet18, y crea un `ColorOcclusionTracker` independiente por cámara. `max_age` y `confirmation_window` se escalan con los FPS reales de cada fuente.

`procesar_videos` lee las tres cámaras en orden de **timestamp corregido** $t=(frame-1)/fps+offset$, sin saltar frames. En cada instante detecta, actualiza cada tracker local y pasa las filas de todas las cámaras juntas a la galería compartida.

**Dónde se va el tiempo.** Perfil medido sobre los videos completos (RTX 4060 Laptop, 3 cámaras 1280×720), en ms por instante:

| Componente | Antes | Ahora | Qué cambió |
|---|---|---|---|
| Re-ID entre cámaras | 67.5 | 4.3 | GPU vía PyTorch; 0.7 recortes por instante en lugar de 1.2 |
| YOLO26m | 30.0 | 25.3 | igual (FP32) |
| Apariencia del tracker local (color + ResNet18) | 13.3 | 5.6 | solo cuando decide algo |
| Resto del tracker y de la asociación | ~16 | ~8 | |
| **Total** | **126** (8 FPS/cámara) | **43** (**23–25 FPS/cámara**) | más rápido que el video (15 FPS) |

Para ir aún más rápido, `CONFIG["detector"]["fp16"] = True` baja a ~35 ms (31 FPS/cámara). En estos videos, sin embargo, la precisión multicámara pasó de 0.99 a 0.95 y el recall de 0.86 a 0.78: las confianzas cambian en milésimas y eso altera decisiones del tracker que están justo en el umbral.

In [ ]:
class MotorLAP01:
    """Un YOLO26m y una ResNet18 compartidos; un seguidor local independiente por cámara."""

    def __init__(self, modelo_dir, config, device="auto", batch=True):
        self.model_dir = Path(modelo_dir).resolve()
        self.config = deepcopy(config)
        if device == "auto":
            device = "cuda:0" if torch.cuda.is_available() else "cpu"
        if device.startswith("cuda") and not torch.cuda.is_available():
            raise RuntimeError("CUDA no está disponible en este kernel. Usa jupyterev con GPU o DEVICE='cpu'.")
        self.device = device
        weights = self.config["weights"]
        for filename in weights.values():
            if not (self.model_dir / filename).is_file():
                raise FileNotFoundError(self.model_dir / filename)
        self.detector = DetectorPersonas(self.model_dir / weights["detector"], self.config["detector"], device, batch)
        self.apariencia = ExtractorResNet18(self.model_dir / weights["appearance"], device)
        appearance = self.config["appearance"]
        gain, offset = (float(v) for v in appearance["calibration"])
        self.mezcla = MezclaApariencia(float(appearance["deep_weight"]), gain, offset)
        self.trackers = {}
        self.detecciones_actuales = {}

    def reiniciar(self, fps):
        reference = float(self.config.get("reference_fps", 15))
        self.trackers, self.detecciones_actuales = {}, {}
        for cid, camera_fps in fps.items():
            settings = dict(self.config["tracker"])
            settings["max_age"] = max(1, round(settings["max_age"] * camera_fps / reference))
            settings["confirmation_window"] = max(settings["min_confirmations"],
                                                  round(settings["confirmation_window"] * camera_fps / reference))
            self.trackers[cid] = ColorOcclusionTracker(self.apariencia, self.mezcla, **settings)

    def procesar(self, frames, source_frames):
        detections = self.detector(frames)
        self.detecciones_actuales = {cid: boxes for cid, (boxes, _) in detections.items()}
        rows = {}
        for cid, frame in frames.items():
            boxes, scores = detections[cid]
            # Se actualiza también sin detecciones: el tiempo de ausencia de los tracks avanza.
            ids = self.trackers[cid].update(frame, boxes, source_frames[cid], scores)
            riesgo = self.trackers[cid].riesgo
            rows[cid] = [{"local_id": int(identity), "x1": float(box[0]), "y1": float(box[1]),
                          "x2": float(box[2]), "y2": float(box[3]), "confidence": float(score),
                          "riesgo": identity in riesgo}
                         for box, score, identity in zip(boxes, scores, ids) if identity > 0]
        return rows

    def calentar(self, frames):
        """Inicializa los kernels antes de medir FPS; no modifica ningún track."""
        self.detector(frames)
        for frame in frames.values():
            self.apariencia(frame, np.array([[0, 0, 32, 64]], dtype=np.float32))
        if self.device.startswith("cuda"):
            torch.cuda.synchronize(self.device)


def crear_asociador(motor, config_camaras):
    encoder = motor.config.get("multicamera_encoder", {"type": "resnet18"})
    if encoder["type"] == "onnx":
        reid = ReIDPersonas(motor.model_dir / encoder["weights"], imgsz=encoder.get("imgsz"),
                            threads=encoder.get("threads", 2), batch_size=encoder.get("batch_size", 8),
                            device=encoder.get("device", motor.device))
    elif encoder["type"] == "resnet18":
        reid = ReIDResNet(motor.apariencia)
    else:
        raise ValueError("multicamera_encoder.type debe ser onnx o resnet18.")
    return AsociadorMulticamara(reid, config_camaras)


@lru_cache(maxsize=4096)
def _color(identity):
    return tuple(int(x) for x in np.random.default_rng(identity).integers(50, 255, 3))


def anotar(frame, camera_id, rows, timestamp_s, display_conf):
    image = frame.copy()
    for row in rows:
        if row["confidence"] < display_conf:
            continue
        x1, y1, x2, y2 = (int(row[key]) for key in ("x1", "y1", "x2", "y2"))
        gid = row.get("global_id")
        color = _color(gid if gid is not None else row["local_id"])
        label = f'G{gid} | {camera_id} L{row["local_id"]}' if gid is not None else f'{camera_id} L{row["local_id"]}'
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(image, label, (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)
    cv2.rectangle(image, (0, 0), (image.shape[1], 38), (22, 22, 22), -1)
    cv2.putText(image, f"{camera_id} | {timestamp_s:.2f}s | {len(rows)} tracks", (12, 26),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
    return image


def crear_mosaico(views, width=960):
    tiles = []
    height = round(width * 9 / 16)
    for view in views.values():
        tile = np.zeros((height, width, 3), dtype=np.uint8)
        scale = min(width / view.shape[1], height / view.shape[0])
        resized = cv2.resize(view, (round(view.shape[1] * scale), round(view.shape[0] * scale)))
        x, y = (width - resized.shape[1]) // 2, (height - resized.shape[0]) // 2
        tile[y:y + resized.shape[0], x:x + resized.shape[1]] = resized
        tiles.append(tile)
    return cv2.vconcat(tiles)


class VistaEnVivo:
    """Actualiza un único mosaico mientras corre la celda (sin widgets)."""

    def __init__(self, ancho=960, camaras=3):
        self.status = display({"text/plain": "Preparando YOLO26m, ResNet18 y Re-ID…"}, raw=True, display_id=True)
        negro = np.zeros((camaras * round(ancho * 9 / 16), ancho, 3), dtype=np.uint8)
        self.picture = display(Image(data=cv2.imencode(".jpg", negro)[1].tobytes()), display_id=True)

    def __call__(self, mosaic, info):
        fps = info["fps_processing"]
        ritmo = "alcanza el ritmo del video" if fps >= info["fps_source"] else "más lento que el video"
        counts = " · ".join(f"{cid}: {count}" for cid, count in info["tracks"].items())
        multi = info["multicamera"]
        self.status.update({"text/plain": (
            f"Frame {info['frame']}/{info['total']} · video {info['timestamp_s']:.1f} s\n"
            f"{fps:.1f} FPS por cámara ({ritmo}) · fuente {info['fps_source']:.0f} FPS · {info['device']}\n"
            f"Tracks activos {counts}\n"
            f"IDs globales: {multi['identidades_globales']} · multicámara: {multi['identidades_multicamara']} "
            f"· modo {multi['mode']}\nDetener: botón ■ de JupyterLab (Interrupt Kernel).")}, raw=True)
        ok, jpeg = cv2.imencode(".jpg", mosaic, [cv2.IMWRITE_JPEG_QUALITY, 80])
        if ok:
            self.picture.update(Image(data=jpeg.tobytes()))

    def terminar(self, resultado):
        r = resultado.resumen
        self.status.update({"text/plain": (
            f"{r['status']} · {r['frames_per_camera']} frames por cámara · {r['fps_per_camera']:.1f} FPS por cámara\n"
            f"IDs globales: {r['multicamera']['identidades_globales']} · "
            f"multicámara: {r['multicamera']['identidades_multicamara']}")}, raw=True)

In [ ]:
COLUMNAS_OBSERVACIONES = ["frame_global", "timestamp_s", "camera_id", "frame", "local_id", "tracklet_id",
                          "global_id", "global_id_online", "x1", "y1", "x2", "y2", "confidence",
                          "X", "Y", "speed", "direction_deg", "projection_valid"]
COLUMNAS_TRAJECTORY_POINTS = ["point_id", "global_id", "timestamp", "camera_id", "local_id", "tracklet_id",
                              "x", "y", "geom", "zone_id", "speed_mps", "direction_deg", "confidence"]


@dataclass
class ResultadoMulticamara:
    """Instantánea de una sesión: no depende del estado del motor ni del asociador tras terminar."""
    resumen: dict
    filas: list
    metadata: dict
    inicios: dict
    offsets: dict
    inicio_grabacion: datetime
    tracklets: list
    identidades: list
    galeria: list
    enlaces: list
    tracklet_uuid: dict
    global_uuid: dict

    def observaciones(self):
        """Salida de la Parte I + II por detección (incluye bbox para auditoría y videos)."""
        return pd.DataFrame(self.filas, columns=COLUMNAS_OBSERVACIONES)

    def trajectory_points(self):
        """Tabla trajectory_points: una fila por global_id, cámara e instante. zone_id = NULL (sin zonas).

        Las pasadas breves que no se unieron a ninguna persona (global_id vacío) no se cuentan ni se exportan.
        """
        obs = self.observaciones()
        obs = obs[obs["global_id"].notna()].reset_index(drop=True)
        x = pd.to_numeric(obs["X"], errors="coerce")
        y = pd.to_numeric(obs["Y"], errors="coerce")
        tabla = pd.DataFrame({
            "point_id": pd.array([pd.NA] * len(obs), dtype="Int64"),   # BIGSERIAL: lo asigna PostgreSQL
            "global_id": obs["global_id"].astype("int64").map(self.global_uuid).astype("string"),
            "timestamp": pd.Timestamp(self.inicio_grabacion) + pd.to_timedelta(obs["timestamp_s"].astype(float), unit="s"),
            "camera_id": obs["camera_id"].astype("string"),
            "local_id": obs["local_id"].astype("int64"),
            "tracklet_id": obs["tracklet_id"].map(self.tracklet_uuid).astype("string"),
            "x": x.astype("float64"),
            "y": y.astype("float64"),
            "geom": pd.array([f"POINT({xi:.4f} {yi:.4f})" if pd.notna(xi) and pd.notna(yi) else pd.NA
                              for xi, yi in zip(x, y)], dtype="string"),
            "zone_id": pd.array([pd.NA] * len(obs), dtype="string"),
            "speed_mps": pd.to_numeric(obs["speed"], errors="coerce").astype("float32"),
            "direction_deg": pd.to_numeric(obs["direction_deg"], errors="coerce").astype("float32"),
            "confidence": obs["confidence"].astype("float32"),
        }, columns=COLUMNAS_TRAJECTORY_POINTS)
        return tabla.sort_values(["timestamp", "camera_id", "local_id"], kind="stable").reset_index(drop=True)

    def guardar_trajectory_points(self, ruta):
        """CSV para \\copy: sin point_id (BIGSERIAL) y con celdas vacías como NULL."""
        ruta = Path(ruta)
        temporal = ruta.with_name(f".{ruta.name}.tmp")
        self.trajectory_points().drop(columns="point_id").to_csv(temporal, index=False)
        temporal.replace(ruta)
        return ruta

    def exportar_videos(self, carpeta, display_conf):
        """Un MP4 por cámara, dibujado con los global_id reconciliados (los mismos de trajectory_points)."""
        carpeta = Path(carpeta)
        carpeta.mkdir(parents=True, exist_ok=True)
        por_frame = {}
        for fila in self.filas:
            por_frame.setdefault((fila["camera_id"], fila["frame"]), []).append(fila)
        rutas = {}
        for cid, meta in self.metadata.items():
            total = self.resumen["frames_by_camera"][cid]
            if not total:
                continue
            destino = carpeta / f"{cid}_procesado.mp4"
            temporal = carpeta / f".{cid}_procesado.tmp.mp4"
            cap, writer = cv2.VideoCapture(meta["video"]), None
            try:
                for _ in range(self.inicios[cid]):
                    if not cap.grab():
                        raise RuntimeError(f"No se pudo avanzar al inicio de {cid}")
                writer = cv2.VideoWriter(str(temporal), cv2.VideoWriter_fourcc(*"mp4v"), meta["fps"],
                                         (meta["width"], meta["height"]))
                if not writer.isOpened():
                    raise RuntimeError(f"No se pudo crear {temporal}")
                for indice in range(total):
                    ok, frame = cap.read()
                    if not ok:
                        break
                    numero = self.inicios[cid] + indice + 1
                    timestamp = (numero - 1) / meta["fps"] + self.offsets[cid]
                    writer.write(anotar(frame, cid, por_frame.get((cid, numero), []), timestamp, display_conf))
            except BaseException:
                if writer is not None:
                    writer.release()
                    writer = None
                temporal.unlink(missing_ok=True)
                raise
            finally:
                cap.release()
                if writer is not None:
                    writer.release()
            temporal.replace(destino)
            rutas[cid] = destino
        return rutas


def procesar_videos(motor, asociador, videos, *, max_frames=None, tiempo_real=True, preview=None,
                    preview_fps=8.0, preview_width=960, start_frames=None, cpu_threads=2, inicio_grabacion=None):
    """Procesa todas las cámaras por timestamp corregido, sin saltar frames.

    start_frames recorta el inicio sin cambiar el reloj; max_frames limita cada cámara por separado y
    las demás continúan si una termina antes. El botón ■ devuelve lo procesado hasta ese momento.
    """
    if max_frames is not None and (type(max_frames) is not int or max_frames < 1):
        raise ValueError("max_frames debe ser None o un entero positivo.")
    if not math.isfinite(preview_fps) or preview_fps <= 0 or cpu_threads < 1 or preview_width < 1:
        raise ValueError("preview_fps, preview_width y cpu_threads deben ser positivos.")
    metadata = {row["camera_id"]: row for row in inspeccionar_videos(videos)}
    if set(asociador.camaras) != set(videos):
        raise ValueError("Modelo/camaras.json no coincide con los videos del dataset.")
    fps = {cid: row["fps"] for cid, row in metadata.items()}
    starts = {cid: 0 for cid in videos}
    starts.update(start_frames or {})
    if set(starts) != set(videos) or any(type(v) is not int or v < 0 for v in starts.values()):
        raise ValueError("INICIOS debe contener las cámaras del dataset con enteros >= 0.")
    offsets = {cid: float(camera.timestamp_offset) for cid, camera in asociador.camaras.items()}
    targets = {cid: row["frames"] - starts[cid] for cid, row in metadata.items()}
    if min(targets.values()) < 1:
        raise ValueError("El inicio configurado queda fuera del video.")
    if max_frames is not None:
        targets = {cid: min(value, max_frames) for cid, value in targets.items()}
    if inicio_grabacion is None:
        inicio_grabacion = datetime.now().astimezone()
    elif isinstance(inicio_grabacion, str):
        inicio_grabacion = datetime.fromisoformat(inicio_grabacion)
    if inicio_grabacion.tzinfo is None:
        inicio_grabacion = inicio_grabacion.astimezone()  # zona horaria del equipo

    session_uuid = uuid.uuid4()
    session_id = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + session_uuid.hex[:6]
    motor.reiniciar(fps)
    asociador.reiniciar(session_uuid)
    asociador.areas = {cid: tuple(row["area"]) for cid, row in metadata.items()}
    display_conf = motor.config["detector"]["display_conf"]
    filas, views = [], {}
    counts = {cid: 0 for cid in videos}
    busy_s, status, start, error = 0.0, "completo", None, None
    previous_preview, first_timestamp, last_timestamp, tick = -math.inf, None, None, 0
    old_threads, old_cv_threads = torch.get_num_threads(), cv2.getNumThreads()
    try:
        torch.set_num_threads(cpu_threads)
        cv2.setNumThreads(1)
        with threadpool_limits(limits=cpu_threads), ExitStack() as stack:
            captures, pending = {}, []
            for cid, row in metadata.items():
                cap = cv2.VideoCapture(row["video"])
                stack.callback(cap.release)
                if not cap.isOpened():
                    raise RuntimeError(f"No se pudo abrir {row['video']}")
                for _ in range(starts[cid]):
                    if not cap.grab():
                        raise RuntimeError(f"No se pudo avanzar al inicio de {cid}")
                captures[cid] = cap
                heapq.heappush(pending, (starts[cid] / fps[cid] + offsets[cid], cid))
            while pending:
                tick_start = time.perf_counter()
                timestamp = pending[0][0]
                selected = []
                while pending and math.isclose(pending[0][0], timestamp, rel_tol=0, abs_tol=1e-8):
                    selected.append(heapq.heappop(pending)[1])
                frames = {}
                for cid in selected:
                    ok, frame = captures[cid].read()
                    if ok:
                        frames[cid] = frame
                    else:
                        status = "fin_anticipado"
                if not frames:
                    continue
                if start is None:
                    motor.calentar(frames)
                    start = tick_start = time.perf_counter()
                    first_timestamp = timestamp
                if tiempo_real:
                    delay = start + timestamp - first_timestamp - time.perf_counter()
                    if delay > 0:
                        time.sleep(delay)
                        tick_start = time.perf_counter()
                source_frames = {cid: starts[cid] + counts[cid] + 1 for cid in frames}
                observations = motor.procesar(frames, source_frames)
                asociador.actualizar(timestamp, frames, observations, occluders=motor.detecciones_actuales)
                for cid, rows in observations.items():
                    for row in rows:
                        filas.append({**row, "frame_global": tick, "timestamp_s": timestamp, "camera_id": cid,
                                      "frame": source_frames[cid], "global_id_online": row["global_id"]})
                    counts[cid] += 1
                    if counts[cid] < targets[cid]:
                        heapq.heappush(pending, ((starts[cid] + counts[cid]) / fps[cid] + offsets[cid], cid))
                if preview is not None and (time.perf_counter() - previous_preview >= 1 / preview_fps or not pending):
                    for cid, frame in frames.items():
                        views[cid] = anotar(frame, cid, observations[cid], timestamp, display_conf)
                    mosaic = crear_mosaico({cid: views.get(cid, np.zeros((metadata[cid]["height"], metadata[cid]["width"], 3),
                                                                       np.uint8)) for cid in videos}, preview_width)
                    current_busy = busy_s + time.perf_counter() - tick_start
                    preview(mosaic, {
                        "frame": sum(counts.values()), "total": sum(targets.values()), "timestamp_s": timestamp,
                        "fps_source": sum(fps.values()) / len(fps),
                        "fps_processing": sum(counts.values()) / len(counts) / max(current_busy, 1e-9),
                        "tracks": {cid: sum(t.frame_seen == starts[cid] + counts[cid] for t in motor.trackers[cid].tracks.values())
                                   for cid in videos},
                        "device": motor.device, "multicamera": asociador.resumen()})
                    previous_preview = time.perf_counter()
                last_timestamp, tick = timestamp, tick + 1
                busy_s += time.perf_counter() - tick_start
    except KeyboardInterrupt:
        status = "interrumpido"
    except Exception as exc:
        status, error = "error", exc
    finally:
        elapsed = time.perf_counter() - start if start is not None else 0.0
        torch.set_num_threads(old_threads)
        cv2.setNumThreads(old_cv_threads)
    if error is not None:
        raise error

    # Reconciliación: cada fila toma su tracklet real (tras los cortes por cambio de persona) y el ID público final.
    numeracion = asociador.numeracion_final()
    for fila in filas:
        track = asociador.resolver(fila["tracklet_id"], fila["timestamp_s"])
        fila["tracklet_id"], fila["global_id"] = track.uid, numeracion.get(track.global_id)
    summary = {"session_id": session_id, "session_uuid": str(session_uuid), "status": status,
               "frames_per_camera": min(counts.values()), "frames_by_camera": counts,
               "frames_total": sum(counts.values()),
               "video_seconds": max((counts[cid] / fps[cid] for cid in videos), default=0),
               "timeline_start_s": first_timestamp, "timeline_end_s": last_timestamp,
               "inicio_grabacion": inicio_grabacion.isoformat(), "wall_seconds": elapsed, "processing_seconds": busy_s,
               "fps_per_camera": sum(counts.values()) / len(counts) / max(busy_s, 1e-9),
               "fps_total": sum(counts.values()) / max(busy_s, 1e-9), "fps_source": fps, "device": motor.device,
               "timestamp_offsets": offsets, "start_frames": starts, "multicamera": asociador.resumen(),
               "cameras": {cid: {**tracker.resumen(), "observations": sum(f["camera_id"] == cid for f in filas),
                                 "frames": counts[cid]} for cid, tracker in motor.trackers.items()}}
    return ResultadoMulticamara(
        resumen=summary, filas=filas, metadata=metadata, inicios=starts, offsets=offsets,
        inicio_grabacion=inicio_grabacion, tracklets=asociador.tabla_tracklets(numeracion),
        identidades=asociador.identidades(numeracion), galeria=asociador.galeria(numeracion),
        enlaces=list(asociador.enlaces),
        tracklet_uuid={uid: t.tracklet_uuid for uid, t in asociador.tracklets.items()},
        global_uuid={publico: asociador.global_uuid(publico) for publico in numeracion.values()})

## 6. Cargar los modelos

Se cargan `yolo26m.pt` (detección), `resnet18-f37072fd.pth` (apariencia local) y `yolo26s-reid.onnx` (Re-ID entre cámaras). El Re-ID corre en la GPU si `jupyterev` tiene `onnx` y `onnx2torch`; si no, usa ONNX Runtime en CPU y el procesamiento es unas 2 veces más lento.

Para usar solo la ResNet18 en la asociación, antes de esta celda cambia `CONFIG["multicamera_encoder"] = {"type": "resnet18"}`. Cada encoder necesita validar sus umbrales con ejemplos etiquetados.

In [ ]:
motor = MotorLAP01(MODELO, CONFIG, device=DEVICE, batch=BATCH_YOLO)
asociador = crear_asociador(motor, CONFIG_CAMARAS)

print("Detector YOLO26m y ResNet18:", motor.device)
print("Encoder multicámara:", motor.config["multicamera_encoder"]["weights"],
      "·", getattr(asociador.reid, "backend", "ResNet18"))
print("Modo multicámara:", asociador.mode)
display(pd.DataFrame({"tracker": pd.Series(motor.config["tracker"]),
                      "association": pd.Series(asociador.config.get("association", {}))}))

## 7. Procesar las tres cámaras

El mosaico muestra `cam01`, `cam02` y `cam03` en vertical. `G7 | cam02 L3` indica identidad global 7 y track local 3 de `cam02`. Los GID en vivo son decisiones *online* y pueden cambiar si más adelante aparece evidencia de fusión o un conflicto. El botón ■ interrumpe el procesamiento y conserva lo ya procesado.

In [ ]:
vista = VistaEnVivo(VISTA_ANCHO, camaras=len(VIDEOS)) if VISTA_EN_VIVO else None
resultado = procesar_videos(
    motor, asociador, VIDEOS,
    max_frames=MAX_FRAMES,
    tiempo_real=TIEMPO_REAL,
    preview=vista,
    preview_fps=VISTA_FPS,
    preview_width=VISTA_ANCHO,
    start_frames=INICIOS,
    cpu_threads=CPU_THREADS,
    inicio_grabacion=INICIO_GRABACION,
)
if vista is not None:
    vista.terminar(resultado)

r = resultado.resumen
print(f"Sesión {r['session_id']} · {r['status']} · {r['frames_total']} frames · {r['video_seconds']:.1f} s de video")
print(f"Capacidad: {r['fps_per_camera']:.2f} FPS por cámara ({r['fps_total']:.2f} FPS totales) en {r['device']}")

## 8. Auditoría

### 8.1 — Seguimiento local por cámara (LAP01)

Mide la fragmentación de IDs (IDs de 1–2 frames y duración mediana), las reapariciones (el mismo `local_id` vuelve tras frames ausente), los solapes entre cajas (IoU ≥ 0.12, es decir, cruces) y los diagnósticos internos del tracker. **No es ground truth:** IDF1 y los cambios de ID requieren anotaciones de referencia.

In [ ]:
observaciones = resultado.observaciones()

auditoria = []
for cid in VIDEOS:
    grupo = observaciones[observaciones.camera_id == cid]
    fila = {"camera_id": cid, **resultado.resumen["cameras"][cid]}
    if not grupo.empty:
        duracion = grupo.groupby("local_id").frame.nunique()
        huecos = np.concatenate([np.diff(np.sort(g.frame.unique())) - 1 for _, g in grupo.groupby("local_id")])
        huecos = huecos[huecos > 0]
        solapes = 0
        for _, cuadro in grupo.groupby("frame"):
            cajas = cuadro[["x1", "y1", "x2", "y2"]].to_numpy()
            solapes += sum(_iou(cajas[i], cajas[j]) >= 0.12 for i in range(len(cajas)) for j in range(i + 1, len(cajas)))
        fila.update(ids_locales=int(grupo.local_id.nunique()), ids_cortos_1_2_frames=int((duracion <= 2).sum()),
                    duracion_mediana_frames=float(duracion.median()), reapariciones=int(len(huecos)),
                    frames_ocultos_max=int(huecos.max()) if len(huecos) else 0, solapes_iou_012=int(solapes))
    auditoria.append(fila)
display(pd.DataFrame(auditoria).set_index("camera_id").T)

### 8.2 — Galería compartida e identidades multicámara

`galeria` muestra lo que las tres cámaras saben de cada persona contada: vistas por cámara y por orientación (frente, espalda, costado, quieto), segundos visible y última cámara donde se vio. `identidades` agrupa los tracklets de cada `global_id`, `tracklets` muestra su intervalo y sus muestras Re-ID, y `enlaces` registra cada fusión (`associate`) y separación (`separate`) con su puntaje. Los puntajes son similitudes, no probabilidades de acierto.

In [ ]:
display(pd.Series(resultado.resumen["multicamera"], name="multicámara"))
print("Galería global compartida (lo que todas las cámaras saben de cada persona contada):")
display(pd.DataFrame(resultado.galeria))

identidades = pd.DataFrame(resultado.identidades)
if not identidades.empty:
    identidades["n_camaras"] = identidades.camaras.map(len)
    identidades = identidades.sort_values(["n_camaras", "global_id"], ascending=[False, True])
display(identidades.head(30))
tracklets = pd.DataFrame(resultado.tracklets)
if not tracklets.empty:
    tracklets = tracklets.sort_values(["global_id", "inicio_s"])
display(tracklets.head(30))
display(pd.DataFrame(resultado.enlaces).tail(30))

## 9. Tabla `trajectory_points`

Cada fila indica dónde estuvo un `global_id` en un instante, después de la homografía y del Re-ID multicámara. Por ahora **no hay zonas**, así que `zone_id` queda `NULL`. Tampoco se generan `zones` ni `spatial_events`.

Las **pasadas breves** que no se unieron a ninguna persona no se cuentan y no se exportan.

| Campo | Origen en este notebook |
|---|---|
| `point_id` | `BIGSERIAL`: lo asigna PostgreSQL (no va en el CSV). |
| `global_id` | UUID v5 de (sesión, GID reconciliado): identidad anónima de sesión. |
| `timestamp` | `INICIO_GRABACION` + timestamp corregido con `timestamp_offset`. |
| `camera_id`, `local_id` | Trazabilidad hacia la cámara y el tracker local. |
| `tracklet_id` | UUID v5 de (sesión, `cam/L/T`). |
| `x`, `y`, `geom` | Punto inferior central de la caja proyectado con `H` (metros); `NULL` sin calibración. |
| `speed_mps`, `direction_deg` | Movimiento del tracklet en el plano (ventana ≥ 0.25 s); dirección en [0, 360). |
| `confidence` | Confianza de YOLO26m para esa detección. |

Como las cámaras se solapan, un mismo `global_id` puede tener varias filas en el mismo instante (una por cámara). La consolidación de duplicados pertenece a la Parte III.

```sql
CREATE TABLE trajectory_points (
    point_id      BIGSERIAL PRIMARY KEY,
    global_id     UUID             NOT NULL,
    "timestamp"   TIMESTAMPTZ      NOT NULL,
    camera_id     VARCHAR(30)      NOT NULL,
    local_id      BIGINT           NOT NULL,
    tracklet_id   UUID             NOT NULL,
    x             DOUBLE PRECISION,
    y             DOUBLE PRECISION,
    geom          geometry(Point),
    zone_id       VARCHAR(50),          -- FK a zones(zone_id) cuando exista la tabla zones
    speed_mps     REAL,
    direction_deg REAL,
    confidence    REAL             NOT NULL
);

\copy trajectory_points (global_id, "timestamp", camera_id, local_id, tracklet_id, x, y, geom, zone_id, speed_mps, direction_deg, confidence) FROM 'Ouput/trajectory_points.csv' WITH (FORMAT csv, HEADER true)
```

In [ ]:
trajectory_points = resultado.trajectory_points()

assert list(trajectory_points.columns) == COLUMNAS_TRAJECTORY_POINTS
assert trajectory_points.zone_id.isna().all()
assert trajectory_points[["global_id", "timestamp", "camera_id", "local_id", "tracklet_id", "confidence"]].notna().all().all()
if not trajectory_points.empty:
    assert trajectory_points.camera_id.str.len().max() <= 30
    assert all(uuid.UUID(v) for v in trajectory_points.global_id.unique())

con_plano = int(trajectory_points.x.notna().sum())
print(f"Filas: {len(trajectory_points)} · global_id: {trajectory_points.global_id.nunique()} · "
      f"tracklets: {trajectory_points.tracklet_id.nunique()} · con x/y: {con_plano}")
if not con_plano:
    print("Sin homografía (modo visual_temporal): x, y, geom, speed_mps y direction_deg quedan NULL. "
          "Calibra las cámaras en la sección 11 para obtener coordenadas del plano.")
display(trajectory_points.dtypes.to_frame("dtype").T)
display(trajectory_points.head(20))

## 10. Exportar

Solo se exporta lo que se va a usar:

- **`Ouput/`**: `cam01_procesado.mp4`, `cam02_procesado.mp4` y `cam03_procesado.mp4`, dibujados con los `global_id` finales, además de `trajectory_points.csv`. Cada ejecución reemplaza los archivos anteriores. No se guardan recortes, embeddings ni frames sueltos.
- **`Modelo/`**: `config_lap01.json` (la configuración usada) y `exportacion.json` (manifiesto SHA-256 de pesos, configuración y notebook, más las versiones de paquetes). `camaras.json` solo se modifica desde la calibración (sección 11).

La exportación del modelo (segunda celda) no depende de haber procesado videos.

In [ ]:
if EXPORTAR_VIDEOS:
    for cid, ruta in resultado.exportar_videos(OUTPUT, motor.config["detector"]["display_conf"]).items():
        print("Video procesado:", ruta.relative_to(BASE))
if EXPORTAR_TRAYECTORIAS:
    print("Tabla:", resultado.guardar_trajectory_points(OUTPUT / "trajectory_points.csv").relative_to(BASE))

In [ ]:
def _sha256(ruta):
    with Path(ruta).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def exportar_modelo(config, config_camaras):
    """Guarda la configuración usada y un manifiesto verificable en Modelo/. No copia ni modifica los pesos."""
    cargar_camaras(config_camaras)  # validar el registro de cámaras antes de escribir
    pesos = set(config["weights"].values())
    encoder = config.get("multicamera_encoder", {})
    if encoder.get("type") == "onnx":
        pesos.add(encoder["weights"])
    faltantes = [nombre for nombre in sorted(pesos) if Path(nombre).name != nombre or not (MODELO / nombre).is_file()]
    if faltantes:
        raise FileNotFoundError(f"Faltan pesos locales en Modelo/: {faltantes}")

    ruta_config = MODELO / "config_lap01.json"
    ruta_config.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n")
    archivos = {f"Modelo/{nombre}": _sha256(MODELO / nombre)
                for nombre in [*sorted(pesos), "config_lap01.json", "camaras.json"]}
    versiones = {}
    for paquete in ("torch", "torchvision", "ultralytics", "numpy", "scipy", "opencv-python", "onnxruntime", "pandas"):
        try:
            versiones[paquete] = importlib.metadata.version(paquete)
        except importlib.metadata.PackageNotFoundError:
            versiones[paquete] = None
    manifiesto = {"version": config.get("version", "2.0"),
                  "fecha": datetime.now().astimezone().isoformat(timespec="seconds"),
                  "notebook": NOTEBOOK.name, "notebook_sha256": _sha256(NOTEBOOK) if NOTEBOOK.is_file() else None,
                  "modo_multicamara": config_camaras.get("mode"), "files_sha256": archivos, "packages": versiones}
    ruta_manifiesto = MODELO / "exportacion.json"
    ruta_manifiesto.write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2) + "\n")
    return ruta_config, ruta_manifiesto


for ruta in exportar_modelo(CONFIG, CONFIG_CAMARAS):
    print("Modelo exportado:", ruta.relative_to(BASE))

## 11. Incorporar la calibración real (homografía)

En cada cámara marca **al menos cuatro puntos del suelo** sobre el video de 1280×720 y sus coordenadas en un plano común en metros. Reserva **al menos dos puntos distintos** para validar. No uses esquinas de paredes ni objetos a otra altura. Si cambia el recorte o la resolución, vuelve a calibrar.

Completa `CALIBRACIONES` para las tres cámaras. La función rechaza puntos degenerados, controles repetidos y errores mayores que el límite. Revisa también en `camaras.json` los polígonos de cobertura, los solapes reales y las transiciones con sus tiempos. Al guardar, el modo pasa a `calibrado`. Luego vuelve a ejecutar desde la sección 2.

In [ ]:
CALIBRACIONES = {}  # No se inventan correspondencias del recinto.
# Formato por cámara (píxeles del video ↔ metros del plano):
# CALIBRACIONES["cam01"] = {
#     "imagen": [[u, v], ...],          "plano": [[X, Y], ...],           # ≥ 4 puntos de ajuste
#     "control_imagen": [[u, v], ...],  "control_plano": [[X, Y], ...],   # ≥ 2 puntos de validación
# }
if CALIBRACIONES:
    if set(CALIBRACIONES) != set(VIDEOS):
        raise ValueError("Completa todas las cámaras antes de activar el modo calibrado.")
    calibrada = deepcopy(CONFIG_CAMARAS)
    for cid, puntos in CALIBRACIONES.items():
        limite = float(calibrada["cameras"][cid].get("max_error_m", 0.75))
        calibrada["cameras"][cid].update(calibrar_homografia(**puntos, max_error_m=limite))
    calibrada["mode"] = "calibrado"
    cargar_camaras(calibrada, VIDEOS)  # validar antes de guardar
    (MODELO / "camaras.json").write_text(json.dumps(calibrada, indent=2, ensure_ascii=False) + "\n")
    CONFIG_CAMARAS = calibrada
    display(pd.DataFrame({cid: cam["validation"] for cid, cam in calibrada["cameras"].items()}).T)
    print("Calibración validada y guardada en Modelo/camaras.json. Vuelve a ejecutar desde la sección 2.")